# Library

In [1]:
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt
import pprint
import numpy as np
from matplotlib.patches import Rectangle, Circle,Polygon
from itertools import permutations
from IPython.display import display, clear_output
import time
import math
import heapq
from lxml import etree
from PIL import Image
import numpy as np
import os
import shutil
import csv

import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Arc, Polygon
import math
import re
from matplotlib.patches import Polygon
import itertools
import matplotlib.patches as patches


import csv
import time
import requests
from urllib.parse import urlparse, parse_qs, urlunparse, urlencode, urljoin
from lxml import html
from requests.adapters import HTTPAdapter, Retry

import os
import csv
import pathlib
import requests
from urllib.parse import urlparse


# Switch to a backend that supports pop-out windows (like Qt5Agg, TkAgg, MacOSX)
# You might need to install the backend if you don't have it (e.g., pip install pyqt5 for Qt5Agg)
matplotlib.use('Agg') # Or try 'TkAgg' or 'MacOSX'

# Schematic (.sch) Extract Function

In [2]:
# Convert XML to dict
def xml_to_dict(element):
    """
    Recursively converts an XML element and its children into a dictionary.
    """
    # Convert attributes and text of the element to a dictionary
    result = {element.tag: {} if element.attrib else None}

    # Add element attributes to the dictionary
    if element.attrib:
        result[element.tag].update((key, value) for key, value in element.attrib.items())

    # Add element text to the dictionary if it exists
    if element.text and element.text.strip():
        text = element.text.strip()
        if result[element.tag]:
            result[element.tag]['text'] = text
        else:
            result[element.tag] = text

    # Convert child elements
    children = list(element)
    if children:
        child_dict = {}
        for child in children:
            child_result = xml_to_dict(child)
            if child.tag in child_dict:
                if isinstance(child_dict[child.tag], list):
                    child_dict[child.tag].append(child_result[child.tag])
                else:
                    child_dict[child.tag] = [child_dict[child.tag], child_result[child.tag]]
            else:
                child_dict.update(child_result)
        if result[element.tag]:
            result[element.tag].update(child_dict)
        else:
            result[element.tag] = child_dict

    return result


def get_deviceset_element(schematic_library, library_name, deviceset_name):
    """
    Extract the entire element of the deviceset from the schematic library.

    Args:
        schematic_library (list): List of libraries from the schematic XML.
        library_name (str): Name of the library containing the deviceset.
        deviceset_name (str): Name of the deviceset to extract.

    Returns:
        dict: The entire element of the deviceset, or None if not found.
    """
    # Find the library with the specified name
    library = next((lib for lib in schematic_library if lib['name'] == library_name), None)
    if not library:
        print(f"Library '{library_name}' not found.")
        return None
    

    # Find the deviceset within the library
    devicesets = library.get('devicesets', {}).get('deviceset', [])
    if not isinstance(devicesets, list):
        devicesets = [devicesets]

    deviceset = next((ds for ds in devicesets if ds['name'] == deviceset_name), None)
    if not deviceset:
        print(f"Deviceset '{deviceset_name}' not found in library '{library_name}'.")
        return None

    return deviceset


def extract_deviceset_info(deviceset_xml, device_name):
    """
    Extract deviceset name, symbol, and package from a deviceset XML.

    Args:
        deviceset_xml (dict): The deviceset XML as a dictionary.
        device_name (str): The name of the device to extract information for.

    Returns:
        dict: A dictionary containing the deviceset name, symbol, and package.
    """
    deviceset_info = {
        'name': deviceset_xml.get('name', 'Unknown'),
        'symbols': 'Unknown',
        'packages': ['Unknown']
    }

    # Extract gates and their symbols
    gates = deviceset_xml.get('gates', {}).get('gate', [])
    if isinstance(gates, dict):
        gates = [gates]
    if gates:
        deviceset_info['symbols'] = gates[0].get('symbol', 'Unknown')

    # Extract devices and their packages
    devices = deviceset_xml.get('devices', {}).get('device', [])
    if isinstance(devices, dict):
        devices = [devices]

    # Find the device with the specified device_name
    device_info = next((device for device in devices if device.get('name') == device_name), None)
    if device_info:
        deviceset_info['packages'] = device_info.get('package', 'Unknown')

    return deviceset_info


def map_part_to_deviceset(schematic_parts, schematic_library, schematic_instance):
    """
    Maps parts to their corresponding deviceset names, libraries, and additional attributes,
    including a list of instances with gate-specific information.

    Args:
        schematic_parts (list): List of parts containing deviceset, library, and value information.
        schematic_instance (list): List of instances containing part names and other attributes.

    Returns:
        dict: A dictionary where keys are part names and values are dictionaries containing deviceset, library,
              device, value, symbol, package, and a list of instances with gate-specific information.
    """
    # Create a mapping of part names to deviceset names, libraries, devices, and values
    # part_to_deviceset_and_library = {
    #     part['name']: {
    #         'deviceset': part['deviceset'],
    #         'library': part.get('library', 'Unknown'),
    #         'device': part.get('device', 'Unknown'),  # Include device information
    #         'value': '' if 'PowerSymbols' in part.get('library', 'Unknown') else part.get('value', ''),  # Conditionally include value information
    #         'package': 'Unknown',  # Initialize package information
    #         'instances': []  # Initialize an empty list for instances
    #     }
    #     for part in schematic_parts
    # }
    part_to_deviceset_and_library = {}

    for part in schematic_parts:
        part_name = part['name']
        part_info = {
            'deviceset': part['deviceset'],
            'library': part.get('library', 'Unknown'),
            'device': part.get('device', 'Unknown'),  # Include device information
            # 'value': '' if 'PowerSymbols' in part.get('library', 'Unknown') else part.get('value', ''),  # Conditionally include value information
            'value': part.get('value', part['deviceset']+part.get('device', 'Unknown')),  # Include value information
            'package': 'Unknown',  # Initialize package information
            'instances': [],  # Initialize an empty list for instances
            'value_in_part': part.get('value', 'Unknown')
        }


        # Optional attributes: add only if they exist
        optional_keys = ['library_urn', 'package3d_urn']
        for key in optional_keys:
            if key in part:
                part_info[key] = part[key]

        part_to_deviceset_and_library[part_name] = part_info
        
    # Map instances to their corresponding parts and add gate-specific information
    for instance in schematic_instance:
        
        part_name = instance['part']
        if part_name in part_to_deviceset_and_library:
            library_name = part_to_deviceset_and_library[part_name]['library']
            deviceset_name = part_to_deviceset_and_library[part_name]['deviceset']
            device_name = part_to_deviceset_and_library[part_name]['device']
            deviceset_element = get_deviceset_element(schematic_library, library_name, deviceset_name)

            # Find the symbol for the instance based on the gate name
            gate_name = instance.get('gate', 'Unknown')
            if deviceset_element is None:
                continue
            gates = deviceset_element.get('gates', {}).get('gate', [])
            if isinstance(gates, dict):
                gates = [gates]
            symbol = next((gate.get('symbol', 'Unknown') for gate in gates if gate.get('name') == gate_name), 'Unknown')

            instance_data = {
                'gate': gate_name,
                'x': instance.get('x', 'Unknown'),
                'y': instance.get('y', 'Unknown'),
                'rot': instance.get('rot', 'R0'),
                'symbol': symbol,
                "attribute": instance.get('attribute', [])
            }
            
            part_to_deviceset_and_library[part_name]['instances'].append(instance_data)

            # Update package information if not already set
            if part_to_deviceset_and_library[part_name]['package'] == 'Unknown':
                deviceset_info = extract_deviceset_info(deviceset_element, device_name)
                part_to_deviceset_and_library[part_name]['package'] = deviceset_info['packages']

    # for part in part_to_deviceset_and_library:
    #     print("part: ",part, part_to_deviceset_and_library[part])
        # print("--------------------------------------------------")

    return part_to_deviceset_and_library


def get_symbol_element_of_instance_from_library(library_xml, part_name, instance_gate_name, part_library):
    """
    Extract the symbol element from the library XML based on the part name and instance gate name.

    Args:
        library_xml (list): List of libraries from the schematic XML.
        part_name (str): Name of the part to extract the symbol for.
        instance_gate_name (str): Name of the gate to match the instance.
        part_library (dict): Dictionary containing part names mapped to their library, symbol, and other details.

    Returns:
        dict: The symbol element, or None if not found.
    """
    # Find the library name and symbol using the part name from the part library
    part_info = part_library.get(part_name)
    if not part_info:
        print(f"Part '{part_name}' not found in part library.")
        return None

    library_name = part_info.get('library')
    instances = part_info.get('instances', [])
    if not library_name or not instances:
        print(f"Library or instances not found for part '{part_name}'.")
        return None

    # Find the library with the specified name
    library = next((lib for lib in library_xml if lib['name'] == library_name), None)
    if not library:
        print(f"Library '{library_name}' not found.")
        return None

    # Extract symbols for the instance matching the specified gate name
    for instance in instances:
        
        if instance.get('gate') != instance_gate_name:
            continue
        

        symbol_name = instance.get('symbol')
        if not symbol_name:
            print(f"Symbol not found for instance in part '{part_name}'.")
            continue

        # Find the symbol in the library
        library_symbols = library.get('symbols', {}).get('symbol', [])
        if not isinstance(library_symbols, list):
            library_symbols = [library_symbols]

        symbol_element = next((sym for sym in library_symbols if sym['name'] == symbol_name), None)
        if not symbol_element:
            print(f"Symbol '{symbol_name}' not found in library '{library_name}'.")
            continue


        
        
        # print("instance:",instance)
        # print("------------------------")
        # Convert the symbol element into the desired dictionary format
        symbol_data = {
            'name': symbol_element.get('name', 'Unknown'),
            'wire': symbol_element.get('wire', []) if isinstance(symbol_element.get('wire', []), list) else [symbol_element.get('wire', [])],
            'text': instance.get('attribute', []) if isinstance(instance.get('attribute', []), list) else [instance.get('attribute', [])],
            'pin': symbol_element.get('pin', []) if isinstance(symbol_element.get('pin', []), list) else [symbol_element.get('pin', [])],
            'rectangle': symbol_element.get('rectangle', []) if isinstance(symbol_element.get('rectangle', []), list) else [symbol_element.get('rectangle', [])],
            'circle': symbol_element.get('circle', []) if isinstance(symbol_element.get('circle', []), list) else [symbol_element.get('circle', [])],
            'polygon': symbol_element.get('polygon', []) if isinstance(symbol_element.get('polygon', []), list) else [symbol_element.get('polygon', [])],
            'attribute': instance.get('attribute', []),
            'symbol_text': symbol_element.get('text',[]) if isinstance(symbol_element.get('text',[]), list) else [symbol_element.get('text',[])]
            
        }

        # print('symbol_data:', symbol_data)
        # if (symbol_element.get('name', 'Unknown') == "I2C_STANDARD"):
        #     import json
        #     with open("example_symbol_element.json", "w") as f:
        #                 json.dump(symbol_data, f, indent=4)
        # print("symbol_data:",symbol_data)
        # print("--------------------------------------------------")
        return symbol_data
    
    print(f"No matching symbol found for gate '{instance_gate_name}' in part '{part_name}'.")
    return None



import json

def find_connects(data, library_name, deviceset_name, device_name=""):
    """
    Extracts the 'connects' field from the specified device in the deviceset.

    Args:
        data (dict): The full JSON-parsed data structure.
        library_name (str): The name of the library (e.g. "40xx").
        deviceset_name (str): The name of the deviceset (e.g. "4066").
        device_name (str): The name of the specific device (e.g. "", "D", "N").

    Returns:
        list of dicts: The list of 'connect' entries, or None if not found.
    """
    for library in data.get("IC_library", []):
        if library.get("name") == library_name:
            devicesets = library.get("devicesets", {}).get("deviceset", {})
            if isinstance(devicesets, list):
                for ds in devicesets:
                    if ds.get("name") == deviceset_name:
                        # import json
                        # with open("text.json", "w") as f:
                        #     json.dump(ds, f, indent=2)
                        devices = ds.get("devices", {}).get("device")
                        if isinstance(devices, dict):
                            devices = [devices]
                        for device in devices:
                            if device.get("name") == device_name:
                                return device.get("connects", {}).get("connect", [])
            elif devicesets.get("name") == deviceset_name:
                devices = devicesets.get("devices", {}).get("device", [])
                if isinstance(devices, dict):
                    devices = [devices]
                for device in devices:
                    if device.get("name") == device_name:
                        return device.get("connects", {}).get("connect", [])
    return None

  
def retrieve_net_info(schematic_net):
    """
    Extract net information from the schematic net data.

    Args:
        schematic_net (list): List of nets from the schematic XML.

    Returns:
        dict: A dictionary where keys are net names and values are lists of segments,
              with each segment containing lists of pinrefs, wires, labels, and junctions.
    """
    net_info = {}

    # Ensure schematic_net is a list
    if not isinstance(schematic_net, list):
        schematic_net = [schematic_net]

    for net in schematic_net:
        net_name = net.get('name', 'Unknown')
        segments = net.get('segment', [])
        if not isinstance(segments, list):
            segments = [segments]

        # Process each segment
        processed_segments = []
        for segment in segments:
            processed_segment = {
                'pinref': segment.get('pinref', []) if isinstance(segment.get('pinref', []), list) else [segment.get('pinref', [])],
                'wire': segment.get('wire', []) if isinstance(segment.get('wire', []), list) else [segment.get('wire', [])],
                'label': segment.get('label', []) if isinstance(segment.get('label', []), list) else [segment.get('label', [])],
                'junction': segment.get('junction', []) if isinstance(segment.get('junction', []), list) else [segment.get('junction', [])]
            }
            processed_segments.append(processed_segment)

        net_info[net_name] = processed_segments

    return net_info



def process_schematic_file(file_path):
    """
    Process the board file and extract relevant information.

    Args:
        file_path (str): Path to the board file.

    Returns:
        dict: A dictionary containing board information.
    """
    tree = ET.parse(file_path)
    root = tree.getroot()
    xml_dict = xml_to_dict(root)



    check_sheet = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get("sheet", [])
    if isinstance(check_sheet, list):
        raise ValueError(f"Multiple sheets found: {len(check_sheet)}")
    
    

    

    # schematic_library = xml_dict['eagle']['drawing']['schematic']['libraries']['library']
    # schematic_parts = xml_dict['eagle']['drawing']['schematic']['parts']["part"]
    # schematic_instance = xml_dict['eagle']['drawing']['schematic']['sheets']["sheet"]['instances']['instance']
    # schematic_net = xml_dict['eagle']['drawing']['schematic']['sheets']["sheet"]['nets']['net']

        # Handle cases where instances, parts, or nets might not exist
    schematic_library = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('libraries', {})
    if schematic_library is None:
        schematic_library = []
    else:
        schematic_library = schematic_library.get("library", []) or []
        if isinstance(schematic_library, dict):
            schematic_library = [schematic_library]

    schematic_parts = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('parts', {})
    if schematic_parts is None:
        schematic_parts = []
    else:
        schematic_parts = schematic_parts.get("part", []) or []
        if isinstance(schematic_parts, dict):
            schematic_parts = [schematic_parts]


    check_busses = (
        xml_dict.get('eagle', {})
                .get('drawing', {})
                .get('schematic', {})
                .get('sheets', {})
                .get('sheet', {})
                .get('busses', {})
    )

    # Extract bus list if available
    schematic_bus = check_busses.get('bus', []) if check_busses else []
    if not isinstance(schematic_bus, list):
        schematic_bus = [schematic_bus]


    schematic_instance = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get("sheet", {}).get('instances', {})
    

    if schematic_instance is None:
        schematic_instance = []
    else:
        schematic_instance = schematic_instance.get("instance", []) or []
        if isinstance(schematic_instance, dict):
            schematic_instance = [schematic_instance]
    
    schematic_net = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get("sheet", {}).get('nets', {})
    if schematic_net is None:
        schematic_net = []
    else:
        schematic_net = schematic_net.get("net", []) or []
        if isinstance(schematic_net, dict):
            schematic_net = [schematic_net]


    schematic_sheets = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get('sheet', {}).get('plain', {})
    schematic_setting = xml_dict.get('eagle', {}).get('drawing', {}).get('grid', {})

    # print("schematic_library:", schematic_library)
    # print("schematic_parts:", schematic_parts)
    # print("schematic_instance:", schematic_instance)
    # print("schematic_net:", schematic_net)



    
    schematic_info = {}
    schematic_info['IC_library'] = schematic_library
    schematic_info['parts'] = map_part_to_deviceset(schematic_parts, schematic_library, schematic_instance)
    schematic_info['nets'] = retrieve_net_info(schematic_net) if schematic_net else {}
    schematic_info['SheetInfo'] = schematic_sheets if schematic_sheets else {}
    schematic_info['setting'] = schematic_setting
    schematic_info['bus'] = schematic_bus
    # for net_name, segments in schematic_info['nets'].items():
    #     print("Net name:", net_name)
    #     print("Segments:")
    #     for segment in segments:
    #         print(segment)
    
    return schematic_info



In [3]:
def process_one_sheet_of_sch(schematic_library,schematic_part,schematic_sheet,schematic_setting):


    schematic_instance = schematic_sheet.get('instances', {})
    

    if schematic_instance is None:
        schematic_instance = []
    else:
        schematic_instance = schematic_instance.get("instance", []) or []
        if isinstance(schematic_instance, dict):
            schematic_instance = [schematic_instance]
    
    schematic_net = schematic_sheet.get('nets', {})
    if schematic_net is None:
        schematic_net = []
    else:
        schematic_net = schematic_net.get("net", []) or []
        if isinstance(schematic_net, dict):
            schematic_net = [schematic_net]


    check_busses = schematic_sheet.get('busses', {})

    # Extract bus list if available
    schematic_bus = check_busses.get('bus', []) if check_busses else []
    if not isinstance(schematic_bus, list):
        schematic_bus = [schematic_bus]

    schematic_sheets = schematic_sheet.get('plain', {})

    # print("schematic_library:", schematic_library)
    # print("schematic_parts:", schematic_parts)
    # print("schematic_instance:", schematic_instance)
    # print("schematic_net:", schematic_net)



    
    schematic_info = {}
    schematic_info['IC_library'] = schematic_library
    schematic_info['parts'] = map_part_to_deviceset(schematic_part, schematic_library, schematic_instance)
    schematic_info['nets'] = retrieve_net_info(schematic_net) if schematic_net else {}
    schematic_info['SheetInfo'] = schematic_sheets if schematic_sheets else {}
    schematic_info['setting'] = schematic_setting
    schematic_info['bus'] = schematic_bus
    # for net_name, segments in schematic_info['nets'].items():
    #     print("Net name:", net_name)
    #     print("Segments:")
    #     for segment in segments:
    #         print(segment)
    
    return schematic_info


In [4]:
def process_schematic_file(file_path):
    """
    Process the board file and extract relevant information.

    Args:
        file_path (str): Path to the board file.

    Returns:
        dict: A dictionary containing board information.
    """
    tree = ET.parse(file_path)
    root = tree.getroot()
    xml_dict = xml_to_dict(root)



    schematic_sheets = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('sheets', {}).get("sheet", [])
    if isinstance(schematic_sheets, dict):
        schematic_sheets = [schematic_sheets]
    
    

    

    # schematic_library = xml_dict['eagle']['drawing']['schematic']['libraries']['library']
    # schematic_parts = xml_dict['eagle']['drawing']['schematic']['parts']["part"]
    # schematic_instance = xml_dict['eagle']['drawing']['schematic']['sheets']["sheet"]['instances']['instance']
    # schematic_net = xml_dict['eagle']['drawing']['schematic']['sheets']["sheet"]['nets']['net']

    # Handle cases where instances, parts, or nets might not exist
    schematic_library = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('libraries', {})
    if schematic_library is None:
        schematic_library = []
    else:
        schematic_library = schematic_library.get("library", []) or []
        if isinstance(schematic_library, dict):
            schematic_library = [schematic_library]

    schematic_parts = xml_dict.get('eagle', {}).get('drawing', {}).get('schematic', {}).get('parts', {})
    if schematic_parts is None:
        schematic_parts = []
    else:
        schematic_parts = schematic_parts.get("part", []) or []
        if isinstance(schematic_parts, dict):
            schematic_parts = [schematic_parts]

    schematic_setting = xml_dict.get('eagle', {}).get('drawing', {}).get('grid', {})

    full_schematic_info = []

    for schematic_sheet in schematic_sheets:

        schematic_info = process_one_sheet_of_sch(schematic_library,schematic_parts,schematic_sheet,schematic_setting)
        full_schematic_info.append(schematic_info)

    return full_schematic_info



# Example

In [ ]:
if __name__ == "__main__":
    # <pinref part="C2" gate="G$1" pin="1"/>
    # <wire x1="53.34" y1="220.98" x2="53.34" y2="210.82" width="0.1524" layer="91"/>
    # Output elements from a file according the path
    # Load and parse the XML file for the specified IC
    folder_path = r"F:\GitHub\IMG2SCH\sample\ArtemisDevKit.sch"
    # folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library"
    # IC_name = "ICM-20948"
    # sch_file_name = f"{folder_path}/{IC_name}/{IC_name}.sch"
    sch_file_name = folder_path
    # print("schematic file name:", sch_file_name)
    # sch_file_name = r"F:\GitHub\PCBEDA\sample PCB\custom\template.sch"
    # sch_file_name = r"F:\GitHub\PCBEDA\sample PCB\custom\temp_add.sch"
    sch_file = "F:\GitHub\IMG2SCH\sample\Arduino-Fio-v23 - 副本.sch"
    # sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\train\sch\weather-bit#SparkFun-Connectors#MICRO_BIT.sch"
    # sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\yolo_training_24\dataset\val\sch\TLC5940-Breakout-v12.sch"
    sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\data\full sch\Arduino_Nano_Every.sch"
    sch_file = r"/Users/taitinglu/Desktop/IMG2SCH/Arduino_Nano_Every copy.sch"
    sch_file = r"/Users/taitinglu/Desktop/IMG2SCH/AWS_IoT_ExpressLink_SARA-R5_Starter_Kit copy.sch"
    schematic_info = process_schematic_file(sch_file)[0]
    

In [ ]:
schematic_info['nets']

# draw schematic 

### Draw Sheet
*Layers: 97 (Info), 94 (Symbols)*

In [5]:
from matplotlib.patches import Arc

def draw_sheet_wire(sheet_wire, alpha=1.0, ax=None):
    """
    Draw a wire (line segment) on the given axes.
    """
    # print("sheet_wire:", sheet_wire)
    x1 = float(sheet_wire['x1'])
    y1 = float(sheet_wire['y1'])
    x2 = float(sheet_wire['x2'])
    y2 = float(sheet_wire['y2'])
    

    layer = sheet_wire.get('layer', '-1')  # Default to layer '-1' if not specified

    color = 'black'
    if layer == '97':
        color = 'gray'
    elif layer == '94':
        color = 'red'
    elif layer == '91':
        color = 'green'

    style = sheet_wire.get('style', 'continuous')
    linestyle = '-'
    if style == 'longdash':
        linestyle = (0, (10, 5))  # long dashes
    elif style == 'shortdash':
        linestyle = (0, (4, 2))   # shorter dashes
    elif style == 'dashdot':
        linestyle = (0, (6, 3, 1, 3))  # dash-dot pattern
    else:
        linestyle = '-'  # continuous

    c = sheet_wire.get('curve', 0)
    curve_deg = float(c)

    if abs(curve_deg) < 1e-9:
        plt.plot([x1, x2], [y1, y2], color = color, linestyle = linestyle)
        return
    chord_len = math.hypot(x2 - x1, y2 - y1) # use original points
    
    if chord_len < 1e-9:
        # Degenerate case: start == end
        return
    
    theta = math.radians(abs(curve_deg))
    R = chord_len / (2.0 * math.sin(theta / 2.0))
    mx_orig = (x1 + x2) / 2.0
    my_orig = (y1 + y2) / 2.0
    d = math.sqrt(R*R - (chord_len / 2.0)**2)
    vx_orig = x2 - x1
    vy_orig = y2 - y1
    nx_orig = -vy_orig
    ny_orig =  vx_orig
    length_n_orig = math.hypot(nx_orig, ny_orig)
    nx_orig /= length_n_orig
    ny_orig /= length_n_orig


    if curve_deg > 0:
        cx_orig = mx_orig + d * nx_orig
        cy_orig = my_orig + d * ny_orig
    else:
        cx_orig = mx_orig - d * nx_orig
        cy_orig = my_orig - d * ny_orig

    cx = cx_orig
    cy = cy_orig

    start_angle = math.degrees(math.atan2(y1 - cy, x1 - cx))
    end_angle   = math.degrees(math.atan2(y2 - cy, x2 - cx))


    def normalize_angle(a):
        return a % 360

    start_angle = normalize_angle(start_angle)
    end_angle   = normalize_angle(end_angle)



    if curve_deg > 0:
        diff = end_angle - start_angle
        if diff < 0:
            diff += 360
        if abs(diff - curve_deg) > 1e-3:
            if diff < curve_deg:
                end_angle += 360
            else:
                end_angle -= 360
    else:
        temp = start_angle
        start_angle = end_angle
        end_angle   = temp

    arc = Arc(
        (cx, cy),           # center
        2*R, 2*R,           # width, height
        angle=0,            # rotation of the whole arc ellipse (0 for a circle)
        theta1=start_angle, # start angle in degrees
        theta2=end_angle,   # end angle in degrees
        edgecolor=color,
        facecolor = color,
        linestyle=linestyle, 
    )


    if ax is None:
        ax = plt.gca()
    # ax.plot([x1, x2], [y1, y2], color=color, alpha=alpha, linestyle=linestyle)
    ax.add_patch(arc)
    
def get_text_alignment(align, rot_angle_deg, is_mirrored):
    """
    Returns horizontal and vertical alignment for text based on align type,
    rotation angle, and optional mirroring.

    Parameters
    ----------
    align : str
        The alignment keyword (e.g., 'bottom-center', 'top-left').
    rot_angle_deg : int
        Rotation angle (0, 90, 180, 270).
    is_mirrored : bool, optional
        Whether the text is mirrored horizontally.

    Returns
    -------
    (ha, va) : tuple
        Horizontal alignment ('left', 'center', 'right')
        and vertical alignment ('top', 'center', 'bottom').
    """

    base_align_map = {
        0:   ('left',  'bottom'),
        90:  ('right', 'bottom'),
        180: ('right', 'top'),
        270: ('left',  'top'),
    }

    align_maps = {
        'bottom-center': {
            0:   ('center', 'bottom'),
            90:  ('right',  'center'),
            180: ('center', 'top'),
            270: ('left',   'center'),
        },
        'bottom-right': {
            0:   ('right', 'bottom'),
            90:  ('right', 'top'),
            180: ('left',  'top'),
            270: ('left',  'bottom'),
        },
        'center-left': {
            0:   ('left',   'center'),
            90:  ('center', 'bottom'),
            180: ('right',  'center'),
            270: ('center', 'top'),
        },
        'center': {
            0:   ('center', 'center'),
            90:  ('center', 'center'),
            180: ('center', 'center'),
            270: ('center', 'center'),
        },
        'center-right': {
            0:   ('right',  'center'),
            90:  ('center', 'top'),
            180: ('left',   'center'),
            270: ('center', 'bottom'),
        },
        'top-left': {
            0:   ('left',  'top'),
            90:  ('left', 'bottom'),
            180: ('right', 'bottom'),
            270: ('right',  'top'),
        },
        'top-center': {
            0:   ('center', 'top'),
            90:  ('left',   'center'),
            180: ('center', 'bottom'),
            270: ('right',  'center'),
        },
        'top-right': {
            0:   ('right', 'top'),
            90:  ('left', 'top'),
            180: ('left',  'bottom'),
            270: ('right', 'bottom'),
        }
    }

    # Default to base map
    ha, va = base_align_map.get(rot_angle_deg % 360, ('center', 'center'))

    # Override if align is in align_maps
    if align in align_maps:
        ha, va = align_maps[align].get(rot_angle_deg % 360, ('center', 'center'))

    # Handle mirrored flip
    exceptions = {
        ('center', 'center'),
        ('center-right', 'center'),
        ('center-left', 'center'),
        ('bottom-center', 'center'),
        ('top-center', 'center'),
    }
    if is_mirrored and (align, ha) not in exceptions:
        ha = 'right' if ha == 'left' else 'left'

    return ha, va


def draw_sheet_text(sheet_text, alpha=1.0, ax=None):


    x = float(sheet_text.get('x', 0))
    y = float(sheet_text.get('y', 0))
    size = float(sheet_text.get('size', 1.0)) * 10
    text = sheet_text.get('text', '')
    layer = sheet_text.get('layer', '-1')
    text_rot = sheet_text.get('rot', 'R0')
    align = sheet_text.get('align', 'bottom-left')
    
    # Color selection
    color = 'black'
    if layer == '97':
        color = 'gray'
    elif layer == '94':
        color = 'red'
    elif layer == '98':
        color = '#d7d769'
    elif layer == '91':
        color = 'green'

    # Rotation and mirroring
    rot_angle_deg = 0.0
    is_mirrored = False
    if text_rot.upper().startswith('R'):
        rot_angle_deg = float(text_rot[1:])
    elif text_rot.upper().startswith('MR'):
        rot_angle_deg = float(text_rot[2:])
        is_mirrored = True

    ha, va = get_text_alignment(align, rot_angle_deg, is_mirrored)
    
    # Keep text upright where needed
    if rot_angle_deg == 180:
        rot_angle_final = 0   # keep upright
    elif rot_angle_deg == 270:
        rot_angle_final = 90
    else:
        rot_angle_final = rot_angle_deg

    if ax is None:
        ax = plt.gca()
    ax.text(x, y, text,
            color=color, fontsize=size,
            rotation=rot_angle_final,
            ha=ha, va=va, alpha=alpha)

    # Draw cross
    cross_size = 0.3
    ax.plot([x - cross_size, x + cross_size], [y, y],
            color=color, linewidth=1, zorder=0)
    ax.plot([x, x], [y + cross_size, y - cross_size],
            color=color, linewidth=1, zorder=0)


def visualize_sheet_texts(sheetinfo_texts,ax=None):
    if not isinstance(sheetinfo_texts, list):
        sheetinfo_texts = [sheetinfo_texts]

    for text in sheetinfo_texts:
        draw_sheet_text(text, alpha=1.0,ax=ax)


def visualize_sheet_wires(sheetinfo_wires,ax=None):
    if not isinstance(sheetinfo_wires, list):
        sheetinfo_wires = [sheetinfo_wires]
        
    for wire in sheetinfo_wires:
        # print("dummy")
        draw_sheet_wire(wire, alpha=0.5,ax=ax)

### Draw Setting-Grid

In [6]:
def visualize_grid_from_setting(setting, ax=None, draw_grid=True):
    """
    Draws a grid on the given matplotlib axes based on the provided setting dictionary.
    Supports 'lines' or 'dots' styles.
    """
    if setting.get('display', 'yes').lower() != 'yes' or not draw_grid:
        return  # Do not draw grid if display is not 'yes'

    if ax is None:
        ax = plt.gca()

    distance = float(setting.get('distance', 5))
    grid_style = setting.get('style', 'lines').lower()
    multiple = int(setting.get('multiple', 1))

    # Grid spacing
    grid_spacing = distance * multiple

    # Get current axis limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()

    if grid_style == 'lines':
        # Draw vertical grid lines
        for x in np.arange(xlim[0], xlim[1] + grid_spacing, grid_spacing):
            ax.axvline(x, color='dimgray', linestyle='-', linewidth=0.5, zorder=0)
        # Draw horizontal grid lines
        for y in np.arange(ylim[0], ylim[1] + grid_spacing, grid_spacing):
            ax.axhline(y, color='dimgray', linestyle='-', linewidth=0.5, zorder=0)

    elif grid_style == 'dots':
        # Create grid of points
        xs = np.arange(xlim[0], xlim[1] + grid_spacing, grid_spacing)
        ys = np.arange(ylim[0], ylim[1] + grid_spacing, grid_spacing)
        X, Y = np.meshgrid(xs, ys)
        ax.scatter(X, Y, color='dimgray', s=5, marker='.', zorder=0)

    else:
        print(f"Unknown grid style '{grid_style}', skipping grid.")


### Bouding box Symbol Comp.

In [7]:
def draw_anchor_box(ax, x, y, ha='center', va='center', width=10, height=5,
                    edgecolor='green', facecolor='none', alpha=1.0, linewidth=2,
                    rot=0, padding=1.0):
    """
    Draw a rectangle with padding. (x, y) is the anchor inside the original box.
    Gray box = original, Green box = padded.

    Parameters
    ----------
    x, y : float
        Anchor point coordinates.
    ha, va : str
        Horizontal/vertical alignment of the anchor in the box.
    width, height : float
        Original box dimensions.
    rot : int
        Rotation angle in degrees; swaps width/height if 90 or 270.
    padding : float
        Extra margin added OUTSIDE the original box.
    show_gray : bool
        Whether to draw the original box in gray for reference.
    """

    rot = rot % 360
    if rot in (90, 270):
        width, height = height, width

    # Horizontal alignment offset
    if ha == 'left':
        offset_x = 0
    elif ha == 'center':
        offset_x = -width / 2
    elif ha == 'right':
        offset_x = -width
    else:
        raise ValueError("ha must be 'left', 'center', or 'right'")

    # Vertical alignment offset
    if va == 'bottom':
        offset_y = 0
    elif va == 'center':
        offset_y = -height / 2
    elif va == 'top':
        offset_y = -height
    else:
        raise ValueError("va must be 'bottom', 'center', or 'top'")

    # Lower-left corner of original box
    lower_left_x = x + offset_x
    lower_left_y = y + offset_y


    # Expand with padding
    lower_left_x -= padding
    lower_left_y -= padding
    width_padded = width + 2 * padding
    height_padded = height + 2 * padding

    # Draw padded green box
    rect_green = patches.Rectangle(
        (lower_left_x, lower_left_y),
        width_padded, height_padded,
        linewidth=linewidth,
        edgecolor=edgecolor,
        facecolor=facecolor,
        alpha=alpha
    )
    ax.add_patch(rect_green)


    return rect_green



def get_text_corners_helper(x, y, ha='center', va='center', width=10, height=5,
                     rot=0, padding=1.0):
    """
    Compute the four corners of a text bounding box with padding.

    Parameters
    ----------
    x, y : float
        Anchor point coordinates.
    ha, va : str
        Horizontal/vertical alignment of the anchor in the box.
    width, height : float
        Original box dimensions.
    rot : int
        Rotation angle in degrees; swaps width/height if 90 or 270.
    padding : float
        Extra margin added OUTSIDE the original box.

    Returns
    -------
    xs, ys : list
        Lists of x and y coordinates of the four corners (in order).
    """

    rot = rot % 360
    if rot in (90, 270):
        width, height = height, width

    # Horizontal alignment offset
    if ha == 'left':
        offset_x = 0
    elif ha == 'center':
        offset_x = -width / 2
    elif ha == 'right':
        offset_x = -width
    else:
        raise ValueError("ha must be 'left', 'center', or 'right'")

    # Vertical alignment offset
    if va == 'bottom':
        offset_y = 0
    elif va == 'center':
        offset_y = -height / 2
    elif va == 'top':
        offset_y = -height
    else:
        raise ValueError("va must be 'bottom', 'center', or 'top'")

    # Lower-left corner of original box
    lower_left_x = x + offset_x
    lower_left_y = y + offset_y

    # Expand with padding
    lower_left_x -= padding
    lower_left_y -= padding
    width_padded = width + 2 * padding
    height_padded = height + 2 * padding

    # Define corners
    corners = np.array([
        [lower_left_x, lower_left_y],
        [lower_left_x + width_padded, lower_left_y],
        [lower_left_x + width_padded, lower_left_y + height_padded],
        [lower_left_x, lower_left_y + height_padded]
    ])

    xs, ys = corners[:, 0].tolist(), corners[:, 1].tolist()
    return xs, ys



def get_text_corners(attr_x, attr_y, align, rot_deg, is_mirrored, text_width, text_height,ax = None):
    """
    Compute bounding box corners of rotated and mirrored text.
    ha and va are used as anchor positions inside the bounding box.

    Parameters
    ----------
    attr_x, attr_y : float
        Anchor position (center of rotation).
    align : str
        Text alignment string (e.g., 'bottom-left', 'top-center', etc.).
    rot_deg : int
        Rotation angle in degrees.
    is_mirrored : bool
        Whether the text is mirrored horizontally.
    text_width, text_height : float
        Size of the text box.

    Returns
    -------
    xs, ys : list of float
        Rotated and transformed bounding box corner coordinates.
    """
    # Draw cross at center
    # cross_size = min(2, 2) * 0.1
    # ax.plot([attr_x - cross_size, attr_x + cross_size], [attr_y, attr_y], color="blue", linewidth=3, alpha=0.2)
    # ax.plot([attr_x, attr_x], [attr_y - cross_size, attr_y + cross_size], color="blue", linewidth=3, alpha=0.2)

    rot_deg = rot_deg % 360
    align = align.lower()

    # # Map alignment to anchor position inside the bounding box
    # ha_va_ratio_map = {
    #     'left':   0.0,
    #     'center': 0.5,
    #     'right':  1.0,
    #     'bottom': 0.0,
    #     'top':    1.0
    # }

    align_maps = {
        'bottom-left':  {0: ('left', 'bottom'), 90: ('right', 'bottom'), 180: ('right', 'top'), 270: ('left', 'top')},
        'bottom-center':{0: ('center','bottom'),90: ('right','center'),180: ('center','top'),270: ('left','center')},
        'bottom-right': {0: ('right','bottom'), 90: ('right','top'),   180: ('left','top'),    270: ('left','bottom')},
        'center-left':  {0: ('left','center'),  90: ('center','bottom'),180: ('right','center'),270: ('center','top')},
        'center':       {0: ('center','center'),90: ('center','center'),180: ('center','center'),270: ('center','center')},
        'center-right': {0: ('right','center'), 90: ('center','top'),   180: ('left','center'),270: ('center','bottom')},
        'top-left':     {0: ('left','top'),     90: ('left','bottom'), 180: ('right','bottom'),270: ('right','top')},
        'top-center':   {0: ('center','top'),   90: ('left','center'), 180: ('center','bottom'),270: ('right','center')},
        'top-right':    {0: ('right','top'),    90: ('left','top'),    180: ('left','bottom'), 270: ('right','bottom')}
    }

    base_align = ('center', 'center')
    if align in align_maps:
        base_align = align_maps[align].get(rot_deg, ('center', 'center'))

    ha_str, va_str = base_align

    # Mirror ha if needed
    exceptions = {
        ('center', 'center'),
        ('center-right', 'center'),
        ('center-left', 'center'),
        ('bottom-center', 'center'),
        ('top-center', 'center'),
    }
    if is_mirrored and (align, ha_str) not in exceptions:
        ha_str = 'right' if ha_str == 'left' else 'left'

    
    # draw_anchor_box(ax, attr_x, attr_y, ha=ha_str, va=va_str,width=text_width, height=text_height,edgecolor="blue",rot=rot_deg,padding=0.5)
    
    xs, ys = get_text_corners_helper(attr_x, attr_y, ha_str, va_str, text_width, text_height, rot_deg, padding=0)

    return xs, ys


In [8]:
def get_wire_bounding_box(wire, x=0, y=0, rot_angle_rad='R0'):
    """
    Compute bounding box of a wire (straight or curved) with rotation and mirror.
    Returns: element_min_x, element_max_x, element_min_y, element_max_y
    """

    # Parse rotation + mirror
    is_mirrored = False
    rot_angle_deg = 0.0
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    theta = math.radians(rot_angle_deg)

    cosθ, sinθ = math.cos(theta), math.sin(theta)
    rot_matrix = [[cosθ, -sinθ], [sinθ, cosθ]]
    mirror_matrix = [[-1, 0], [0, 1]] if is_mirrored else [[1, 0], [0, 1]]

    def transform_point(px, py):
        # rotate
        rx = px * rot_matrix[0][0] + py * rot_matrix[0][1]
        ry = px * rot_matrix[1][0] + py * rot_matrix[1][1]
        # mirror
        mx = rx * mirror_matrix[0][0] + ry * mirror_matrix[0][1]
        my = rx * mirror_matrix[1][0] + ry * mirror_matrix[1][1]
        # translate
        return x + mx, y + my

    # Endpoints
    x1_orig, y1_orig = float(wire['x1']), float(wire['y1'])
    x2_orig, y2_orig = float(wire['x2']), float(wire['y2'])
    x1, y1 = transform_point(x1_orig, y1_orig)
    x2, y2 = transform_point(x2_orig, y2_orig)

    curve_deg = float(wire.get('curve', 0))

    # Straight wire
    if abs(curve_deg) < 1e-9:
        return min(x1, x2), max(x1, x2), min(y1, y2), max(y1, y2)

    # Arc case
    chord_len = math.hypot(x2_orig - x1_orig, y2_orig - y1_orig)
    if chord_len < 1e-9:
        return x1, x1, y1, y1

    theta_arc = math.radians(abs(curve_deg))
    R = chord_len / (2.0 * math.sin(theta_arc / 2.0))
    mx_orig = (x1_orig + x2_orig) / 2.0
    my_orig = (y1_orig + y2_orig) / 2.0
    d = math.sqrt(R*R - (chord_len/2.0)**2)

    vx_orig, vy_orig = x2_orig - x1_orig, y2_orig - y1_orig
    nx_orig, ny_orig = -vy_orig, vx_orig
    length_n = math.hypot(nx_orig, ny_orig)
    nx_orig, ny_orig = nx_orig/length_n, ny_orig/length_n

    if curve_deg > 0:
        cx_orig = mx_orig + d * nx_orig
        cy_orig = my_orig + d * ny_orig
    else:
        cx_orig = mx_orig - d * nx_orig
        cy_orig = my_orig - d * ny_orig

    cx, cy = transform_point(cx_orig, cy_orig)

    # Angles of endpoints
    start_angle = math.degrees(math.atan2(y1 - cy, x1 - cx)) % 360
    end_angle   = math.degrees(math.atan2(y2 - cy, x2 - cx)) % 360

    if curve_deg > 0:
        diff = end_angle - start_angle
        if diff < 0: diff += 360
        if abs(diff - curve_deg) > 1e-3:
            if diff < curve_deg:
                end_angle += 360
            else:
                end_angle -= 360
    else:
        start_angle, end_angle = end_angle, start_angle

    def point_on_circle(angle_deg):
        rad = math.radians(angle_deg)
        return cx + R * math.cos(rad), cy + R * math.sin(rad)

    # Points: start, end, and possible extrema at 0/90/180/270
    arc_points = [point_on_circle(start_angle), point_on_circle(end_angle)]
    for ang in [0, 90, 180, 270]:
        norm_ang = ang % 360
        norm_s, norm_e = start_angle % 360, end_angle % 360
        if start_angle < end_angle:
            in_range = norm_s <= norm_ang <= norm_e
        else:
            in_range = norm_ang >= norm_s or norm_ang <= norm_e
        if in_range:
            arc_points.append(point_on_circle(ang))

    xs, ys = zip(*arc_points)
    return min(xs), max(xs), min(ys), max(ys)


def get_rectangle_bounding_box(rect_data, x=0, y=0, rot_angle_rad='R0'):
    """
    Compute the bounding box of a rectangle with rotation.

    Parameters
    ----------
    rect_data : dict
        Dictionary with 'x1','y1','x2','y2' defining opposite corners.
    x, y : float
        Translation offsets.
    rot_angle_rad : str
        Rotation string like 'R0', 'R90', 'MR0', etc.

    Returns
    -------
    element_min_x, element_max_x, element_min_y, element_max_y
    """
    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    theta = math.radians(rot_angle_deg)

    # Extract rectangle corners
    x1 = float(rect_data['x1'])
    y1 = float(rect_data['y1'])
    x2 = float(rect_data['x2'])
    y2 = float(rect_data['y2'])

    # All four corners before rotation
    corners = [
        (x1, y1),
        (x1, y2),
        (x2, y1),
        (x2, y2)
    ]

    rotated_corners = []
    for cx, cy in corners:
        # Rotate
        rx = x + (cx * math.cos(theta) - cy * math.sin(theta))
        ry = y + (cx * math.sin(theta) + cy * math.cos(theta))
        # Mirror horizontally if needed
        if is_mirrored:
            rx = 2*x - rx
        rotated_corners.append((rx, ry))

    xs, ys = zip(*rotated_corners)
    return min(xs), max(xs), min(ys), max(ys)


def get_pin_bounding_box(pin, x=0, y=0, rot_angle_rad='R0'):
    """
    Compute bounding box for a single pin with rotation & mirroring.
    """
    # Parse rotation and mirror
    is_mirrored = False
    rot_angle_deg = 0.0
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    rot_angle_rad = math.radians(rot_angle_deg)

    # Length mapping
    length_map = {
        'point': 0.0,
        'short': 2.54,
        'middle': 2.54 * 2,
        'long': 2.54 * 3
    }

    # Base
    x_base = float(pin['x'])
    y_base = float(pin['y'])

    # Rotate base
    rotated_x_base = x + (x_base * math.cos(rot_angle_rad) - y_base * math.sin(rot_angle_rad))
    rotated_y_base = y + (x_base * math.sin(rot_angle_rad) + y_base * math.cos(rot_angle_rad))

    # Extension
    length_type = pin.get('length', 'long').lower()
    extension = length_map.get(length_type, 2.54)

    # Pin angle
    rot_pin_str = pin.get('rot', 'R0')
    pin_angle_deg = float(rot_pin_str[1:]) if rot_pin_str.upper().startswith('R') else float(rot_pin_str[2:])
    pin_angle_rad = math.radians(pin_angle_deg) + rot_angle_rad

    # End point
    x_end = rotated_x_base + extension * math.cos(pin_angle_rad)
    y_end = rotated_y_base + extension * math.sin(pin_angle_rad)

    # Apply mirroring consistently to both base and end
    if is_mirrored:
        rotated_x_base = 2*x - rotated_x_base
        x_end = 2*x - x_end

    # Collect points
    points = [(rotated_x_base, rotated_y_base), (x_end, y_end)]

    # Dotted pins: add a small circle radius
    if pin.get('function', '') == 'dot':
        circle_radius = 1.0
        points.extend([
            (x_end + circle_radius, y_end + circle_radius),
            (x_end - circle_radius, y_end - circle_radius)
        ])

    xs, ys = zip(*points)
    return min(xs), max(xs), min(ys), max(ys)


def get_symbol_text_bounding_box(attribute, x=0, y=0, rot_angle_rad='R0', ax=None):
    """
    Compute the bounding box for a single text attribute.
    Considers symbol rotation, text local rotation, mirroring, alignment,
    and multiline text.
    Returns: element_min_x, element_max_x, element_min_y, element_max_y
    """

    if not attribute or attribute.get('display', 'on').lower() == 'off':
        return None

    # Attribute fields
    attr_name = attribute.get('text', '')
    attr_local_x = float(attribute.get('x', 0))
    attr_local_y = float(attribute.get('y', 0))
    attr_size = float(attribute.get('size', 1.0))
    attr_align = str(attribute.get('align', 'bottom-left')).lower()
    attr_rot_str = str(attribute.get('rot', 'R0')).upper()

    # Parse local text rotation
    local_mirror = False
    local_rot_deg = 0.0
    if attr_rot_str.startswith('MR'):
        local_mirror = True
        local_rot_deg = float(attr_rot_str[2:]) if len(attr_rot_str) > 2 else 0.0
    elif attr_rot_str.startswith('R'):
        local_rot_deg = float(attr_rot_str[1:]) if len(attr_rot_str) > 1 else 0.0

    # Parse symbol rotation
    is_mirrored = False
    symbol_rot_deg = 0.0
    rot_angle_str = str(rot_angle_rad).upper()
    if rot_angle_str.startswith('MR'):
        is_mirrored = True
        symbol_rot_deg = float(rot_angle_str[2:]) if len(rot_angle_str) > 2 else 0.0
    elif rot_angle_str.startswith('R'):
        symbol_rot_deg = float(rot_angle_str[1:]) if len(rot_angle_str) > 1 else 0.0

    # Combined mirror: XOR of symbol and local
    final_mirror = is_mirrored ^ local_mirror
    final_rot_deg = (symbol_rot_deg + local_rot_deg) % 360

    theta = math.radians(symbol_rot_deg)
    cos_t, sin_t = math.cos(theta), math.sin(theta)

    # Rotate + mirror local text coordinates into global
    rx = attr_local_x * cos_t - attr_local_y * sin_t
    ry = attr_local_x * sin_t + attr_local_y * cos_t
    if is_mirrored:
        rx = -rx
    attr_global_x = x + rx
    attr_global_y = y + ry

    # Handle multiline text
    lines = attr_name.splitlines()
    num_lines = len(lines)
    max_line_len = max((len(line) for line in lines), default=0)

    # Estimate text dimensions
    fontsize = attr_size * 10
    text_width = int(max_line_len) * fontsize * 0.6 / 10.0
    text_height = int(num_lines) * fontsize * 1.2 / 10.0

    # Compute bounding box corners
    xs, ys = get_text_corners(
        attr_global_x, attr_global_y,
        attr_align, final_rot_deg, final_mirror,
        text_width, text_height, ax
    )

    return min(xs), max(xs), min(ys), max(ys)

def get_attribute_text_bounding_box(attribute, part_name="Unknown", part_value="Unknown", symbol_name="Unknown",ax=None):
    """
    Compute bounding box for a single attribute text with rotation and mirroring.
    Handles tricky cases like rot=90 with bottom-left align.
    Returns (min_x, max_x, min_y, max_y).
    """

    if not attribute or attribute.get('display', 'on').lower() == 'off':
        return None

    # Attribute fields
    attr_name = attribute.get('name', '')
    attr_x = float(attribute.get('x', 0))
    attr_y = float(attribute.get('y', 0))
    attr_size = float(attribute.get('size', 1.0))
    attr_rot = str(attribute.get('rot', 'R0')).upper()
    attr_align = str(attribute.get('align', 'bottom-left')).lower()

    # Mirroring and rotation
    is_mirrored = attr_rot.startswith('MR')
    rot_val = attr_rot[2:] if is_mirrored else attr_rot[1:]
    try:
        rot_deg = int(rot_val)
    except ValueError:
        rot_deg = 0
    rot_deg = rot_deg % 360

    # Decide text to draw
    if attr_name == "NAME" and part_name != "Unknown":
        display_text = part_name
    elif attr_name == "VALUE":
        display_text = part_value if part_value not in ["Unknown", ""] else symbol_name
        if 'GND' in part_name.upper():
            display_text = re.sub(r'\d+', '', part_name)
    else:
        if symbol_name.upper() == "GND":
            return None
        display_text = attr_name

    if not display_text:
        return None

    # Estimate unrotated text dimensions
    fontsize = attr_size * 10
    text_width = len(display_text) * fontsize * 0.6 / 10.0
    text_height = fontsize * 1.2 / 10.0

    xs, ys = get_text_corners(attr_x, attr_y, attr_align, rot_deg, is_mirrored, text_width, text_height,ax)
    
    return min(xs), max(xs), min(ys), max(ys)

def get_circle_bounding_box(circle_data, x=0, y=0, rot_angle_rad='R0'):
    """
    Compute the bounding box of a circle with rotation and mirroring.

    Returns
    -------
    element_min_x, element_max_x, element_min_y, element_max_y
    """
    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    elif rot_angle_rad.upper().startswith('MR'):
        rot_angle_deg = float(rot_angle_rad[2:])
        is_mirrored = True
    rot_angle_rad = math.radians(rot_angle_deg)

    # Extract circle parameters
    x_center = float(circle_data['x'])
    y_center = float(circle_data['y'])
    radius   = float(circle_data['radius'])

    # Rotate and mirror center
    rotated_x_center = x + (x_center * math.cos(rot_angle_rad) - y_center * math.sin(rot_angle_rad))
    rotated_y_center = y + (x_center * math.sin(rot_angle_rad) + y_center * math.cos(rot_angle_rad))

    if is_mirrored:
        rotated_x_center = 2*x - rotated_x_center

    # Bounding box
    element_min_x = rotated_x_center - radius
    element_max_x = rotated_x_center + radius
    element_min_y = rotated_y_center - radius
    element_max_y = rotated_y_center + radius

    return element_min_x, element_max_x, element_min_y, element_max_y

def get_polygon_bounding_box(polygons, x=0, y=0, rot_angle_rad='R0'):
    """
    Compute the bounding box for a list of polygons with rotation and mirroring.
    
    Parameters
    ----------
    polygons : list
        Each dict has 'vertex': list of {'x': str, 'y': str}
    x, y : float
        Anchor position for the symbol instance.
    rot_angle_rad : str
        Rotation string like 'R0', 'R90', 'MR0', 'MR90'.
        
    Returns
    -------
    element_min_x, element_max_x, element_min_y, element_max_y : float
        The bounding box coordinates of the polygon set.
    """

    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    theta = math.radians(rot_angle_deg)

    cos_t, sin_t = math.cos(theta), math.sin(theta)

    all_x, all_y = [], []


    vertices = polygons.get('vertex', [])
    for v in vertices:
        px = float(v['x'])
        py = float(v['y'])

        # Apply rotation
        rx = px * cos_t - py * sin_t
        ry = px * sin_t + py * cos_t

        # Apply mirroring across vertical axis
        if is_mirrored:
            rx = -rx

        # Translate
        rotated_x = x + rx
        rotated_y = y + ry

        all_x.append(rotated_x)
        all_y.append(rotated_y)

    if not all_x or not all_y:
        return None  # no vertices

    return min(all_x), max(all_x), min(all_y), max(all_y)




### Bounding box Symbol

In [9]:
def draw_one_box(x_center, y_center, length, width, ax, color='orange', thickness=1.5):
    x_min = x_center - length / 2 
    y_min = y_center - width / 2 

    rect = Rectangle(
        (x_min, y_min), length, width,
        linewidth=thickness, edgecolor=color, facecolor='none'
    )
    ax.add_patch(rect)

    # Draw cross at center
    cross_size = min(length, width) * 0.1
    ax.plot([x_center - cross_size, x_center + cross_size], [y_center, y_center], color=color, linewidth=1.2)
    ax.plot([x_center, x_center], [y_center - cross_size, y_center + cross_size], color=color, linewidth=1.2)


def get_bounding_box_for_symbol_instance2(symbol_element, instance_x, instance_y, rot="R0", part_name="Unknown", part_value="Unknown",symbol_name="Unknown",space=10,ax=None):
    
    """
    Calculates the bounding box (xlim, ylim) for a single symbol instance.
    (No changes in this function)
    """
    # print("get_bounding_box_for_symbol_instance2")

        
    min_x_symbol = float('inf')
    max_x_symbol = float('-inf')
    min_y_symbol = float('inf')
    max_y_symbol = float('-inf')

    rot_angle_deg = 0.0
    if rot.upper().startswith('R180'):
        rot_angle_deg = 0
    elif rot.upper().startswith('R270'):
        rot_angle_deg = 90
    elif rot.upper().startswith('R'):
        rot_angle_deg = float(rot[1:])
    elif rot.upper().startswith('MR'):
        rot_angle_deg = 360 - float(rot[2:])
    rot_angle_rad = math.radians(rot_angle_deg)
        

    elements_list = []
    for key in ['pin', 'wire', 'rectangle', 'circle', 'polygon', 'text', 'symbol_text']:
        for element in symbol_element.get(key, []):
            element_with_type = element.copy()
            element_with_type['element_type'] = key
            elements_list.append([element_with_type])

    # return elements_list

    # print("elements_list:", elements_list)
    for elements in elements_list:
        # print("elements:", elements)
        for element in elements:
            # print("element:", element)
            element_type = element.get('element_type', 'unknown')

            # print(element_type," element:", element)
            element_min_x = float('inf')
            element_max_x = float('-inf')
            element_min_y = float('inf')
            element_max_y = float('-inf')

            if element_type == 'pin':
                element_min_x, element_max_x, element_min_y, element_max_y = get_pin_bounding_box(element, instance_x, instance_y, rot)

            elif element_type == 'wire':
                element_min_x, element_max_x, element_min_y, element_max_y = get_wire_bounding_box(element, instance_x, instance_y, rot)
            

            elif element_type == 'rectangle':
                element_min_x, element_max_x, element_min_y, element_max_y = get_rectangle_bounding_box(element, instance_x, instance_y, rot)


            elif element_type == 'circle':
                element_min_x, element_max_x, element_min_y, element_max_y = get_circle_bounding_box(element, instance_x, instance_y, rot)


            elif element_type == 'polygon':
                element_min_x, element_max_x, element_min_y, element_max_y = get_polygon_bounding_box(element, instance_x, instance_y, rot)

            
            elif (element_type == 'text' and element.get('display', 'on').lower() != 'off'):
                element_min_x, element_max_x, element_min_y, element_max_y = get_attribute_text_bounding_box(element,part_name, part_value, symbol_name,ax)


            elif (element_type == 'symbol_text' and element.get('display', 'on').lower() != 'off' and not element.get('text', '').upper().startswith('>')):
                # print("symbol_text: ",element)
                element_min_x, element_max_x, element_min_y, element_max_y = get_symbol_text_bounding_box(element, instance_x, instance_y, rot)




            min_x_symbol = min(min_x_symbol, element_min_x)
            max_x_symbol = max(max_x_symbol, element_max_x)
            min_y_symbol = min(min_y_symbol, element_min_y)
            max_y_symbol = max(max_y_symbol, element_max_y)
        

            # if element_type in ['circle']:
            #     # Draw bounding box for each element if ax is provided
            #     if element_min_x != float('inf') and element_max_x != float('-inf'):
            #         x_center = (element_min_x + element_max_x) / 2
            #         y_center = (element_min_y + element_max_y) / 2
            #         length = element_max_x - element_min_x
            #         width = element_max_y - element_min_y
            #         draw_one_box(x_center, y_center, length+1, width+1, ax, color="#CE5192", thickness=2)
            

                    
       
    # print("---------------------------------")

    if min_x_symbol == float('inf'): # No elements found
        return ((instance_x, instance_x + 200), (instance_y, instance_y + 200)) # Return a default range around instance position
    else:
        xlim = (min_x_symbol-space, max_x_symbol+space)
        ylim = (min_y_symbol-space, max_y_symbol+space)



            
        return xlim, ylim, elements_list


def draw_element_list(element_lists, draw_list,ax = None):
    for element_list in element_lists:
        instance_x = element_list['x_instance']
        instance_y = element_list['y_instance']
        rot = element_list['rot']
        ele_list = element_list['list']
        min_x_symbol = float('inf')
        max_x_symbol = float('-inf')
        min_y_symbol = float('inf')
        max_y_symbol = float('-inf')
        part_name = element_list['part_name']
        part_value = element_list['part_value']
        symbol_name = element_list['symbol_name']
        for elements in ele_list:
            # print("elements:", elements)
            for element in elements:
                # print("element:", element)
                element_type = element.get('element_type', 'unknown')

                # print(element_type," element:", element)
                element_min_x = float('inf')
                element_max_x = float('-inf')
                element_min_y = float('inf')
                element_max_y = float('-inf')

                if element_type == 'pin':
                    element_min_x, element_max_x, element_min_y, element_max_y = get_pin_bounding_box(element, instance_x, instance_y, rot)

                elif element_type == 'wire':
                    element_min_x, element_max_x, element_min_y, element_max_y = get_wire_bounding_box(element, instance_x, instance_y, rot)
                

                elif element_type == 'rectangle':
                    element_min_x, element_max_x, element_min_y, element_max_y = get_rectangle_bounding_box(element, instance_x, instance_y, rot)


                elif element_type == 'circle':
                    element_min_x, element_max_x, element_min_y, element_max_y = get_circle_bounding_box(element, instance_x, instance_y, rot)


                elif element_type == 'polygon':
                    element_min_x, element_max_x, element_min_y, element_max_y = get_polygon_bounding_box(element, instance_x, instance_y, rot)

                
                elif (element_type == 'text' and element.get('display', 'on').lower() != 'off'):
                    element_min_x, element_max_x, element_min_y, element_max_y = get_attribute_text_bounding_box(element,part_name, part_value, symbol_name,ax)


                elif (element_type == 'symbol_text' and element.get('display', 'on').lower() != 'off' and not element.get('text', '').upper().startswith('>')):
                    # print("symbol_text: ",element)
                    element_min_x, element_max_x, element_min_y, element_max_y = get_symbol_text_bounding_box(element, instance_x, instance_y, rot)




                min_x_symbol = min(min_x_symbol, element_min_x)
                max_x_symbol = max(max_x_symbol, element_max_x)
                min_y_symbol = min(min_y_symbol, element_min_y)
                max_y_symbol = max(max_y_symbol, element_max_y)
            

                if element_type in draw_list:
                    # Draw bounding box for each element if ax is provided
                    if element_min_x != float('inf') and element_max_x != float('-inf'):
                        x_center = (element_min_x + element_max_x) / 2
                        y_center = (element_min_y + element_max_y) / 2
                        length = element_max_x - element_min_x
                        width = element_max_y - element_min_y
                        draw_one_box(x_center, y_center, length+1, width+1, ax, color="#CE5192", thickness=2)



### get bounding box for full sch

In [10]:
def get_bounding_box_full_sch(boxes):
    if not boxes:
        return None  

    # Initialize with the first box limits
    min_x = float(boxes[0]['x_center']) - float(boxes[0]['length'])/2
    max_x = float(boxes[0]['x_center']) + float(boxes[0]['length'])/2
    min_y = float(boxes[0]['y_center']) - float(boxes[0]['width'])/2
    max_y = float(boxes[0]['y_center']) + float(boxes[0]['width'])/2
    for box in boxes[1:]:
        min_x = min(min_x, float(box['x_center']) - float(box['length'])/2)
        max_x = max(max_x, float(box['x_center']) + float(box['length'])/2)
        min_y = min(min_y, float(box['y_center']) - float(box['width'])/2)
        max_y = max(max_y, float(box['y_center']) + float(box['width'])/2)

    # Construct final bounding box
    final_box = {
        'part_name': 'SheetInfo',
        'net_name': 'final_bounding_box',
        'x_center': (min_x + max_x) / 2,
        'y_center': (min_y + max_y) / 2,
        'length': max_x - min_x,
        'width': max_y - min_y,
    }

    return [final_box]

In [11]:

def get_list_of_symbol_bounding_boxes(schematic_info,space=5):
    """
    Returns a list of bounding box information for each symbol instance in the schematic.

    Args:
        schematic_info (dict): The schematic information dictionary containing 'parts', 'nets', and 'IC_library'.

    Returns:
        list: A list of dictionaries, each containing:
              {
                  'part_name': str,
                  'gate_instance': str,
                  'xlim': (min_x, max_x),
                  'ylim': (min_y, max_y)
              }
    """
    parts_info = schematic_info.get('parts')
    ic_library = schematic_info.get('IC_library')

    list_of_bounding_boxes = []
    element_lists = []

    # --- Calculate bounding box for parts using get_bounding_box_for_symbol_instance ---
    if parts_info:
        for part_name, part_data in parts_info.items():
            # print(f"Processing part '{part_name}': {part_data}")
            part_value = part_data.get('value', 'Unknown')
            
            for instance in part_data.get('instances', []):
                gate_instance = instance.get('gate', '') # Handle cases where gate might be missing
                symbol_element = get_symbol_element_of_instance_from_library(
                    ic_library, part_name, gate_instance, parts_info
                )
                

                symbol_name = instance['symbol']

                if symbol_element:
                    rot_instance = instance['rot']
                    x_instance = float(instance['x'])
                    y_instance = float(instance['y'])

                    symbol_instance_bbox = get_bounding_box_for_symbol_instance2(
                        symbol_element, x_instance, y_instance, rot_instance,part_name, part_value, symbol_name, space=space
                    )
                    xlim_instance, ylim_instance, element_list = symbol_instance_bbox

                    ele_list_pack = {}
                    ele_list_pack['list'] = element_list
                    
                    ele_list_pack['x_instance'] = x_instance
                    ele_list_pack['y_instance'] = y_instance
                    ele_list_pack['rot'] = rot_instance
                    ele_list_pack['part_name'] = part_name
                    ele_list_pack['part_value'] = part_value
                    ele_list_pack['symbol_name'] = symbol_name

                    element_lists.append(ele_list_pack)

                    length = xlim_instance[1] - xlim_instance[0]
                    width = ylim_instance[1] - ylim_instance[0]

                    lib         = part_data['library']
                    symbol       = instance['symbol']
                    # base_name    = os.path.splitext(os.path.basename(sch_file))[0]
                    key          = f"{lib}#{symbol}"

                    bbox_info = {
                        'part_name': part_name,
                        'gate_instance': gate_instance,
                        'x_center': (xlim_instance[0] + xlim_instance[1]) / 2,
                        'y_center': (ylim_instance[0] + ylim_instance[1]) / 2,
                        'length': length,
                        'width': width,
                        'key':key,
                        'cat':'symbol'
                    }

                    list_of_bounding_boxes.append(bbox_info)

    return list_of_bounding_boxes, element_lists


def draw_list_of_bounding_boxes(bounding_boxes, ax, color='orange'):
    """
    Draws bounding boxes for each symbol instance on the given Matplotlib axis,
    and marks the center (x_center, y_center) with a cross.

    Args:
        bounding_boxes (list): A list of bounding box dictionaries with keys:
            'part_name', 'gate_instance', 'length', 'width', 'x_center', 'y_center'.
        ax (matplotlib.axes.Axes): The axis to draw the bounding boxes on.
        color (str): The color of the bounding box (default: 'orange').
    """
    for bbox in bounding_boxes:
        x_center = bbox['x_center']
        y_center = bbox['y_center']
        length = bbox['length']
        width = bbox['width']


        # Print part name, gate instance, center coordinates and dimensions for each bounding box
        # print(f"Part: {bbox['part_name']}, Instance: {bbox['gate_instance']}, Center: ({x_center:.2f}, {y_center:.2f}), Length: {length:.2f}, Width: {width:.2f}")

        draw_one_box(
            x_center, y_center, length, width, ax, color=color
        )

### Bounding box Sheet

In [12]:
def get_one_sheet_text_bounding_box(element, ax=None):
    """
    Compute the bounding box for a sheet text element.
    Handles multiline text, rotation, alignment, and mirroring.

    Parameters
    ----------
    element : dict
        Dictionary with keys:
          - 'text': the text string (may contain newlines)
          - 'x', 'y': coordinates (strings or floats)
          - 'size': text size scale
          - 'align': alignment (e.g., 'bottom-left', 'center')
          - 'rot': rotation string (e.g., 'R0', 'R90', 'MR0', 'MR90')
    ax : matplotlib axis (optional)
        Axis, if you want to debug by drawing.

    Returns
    -------
    element_min_x, element_max_x, element_min_y, element_max_y
    """

    # Extract fields safely
    text = str(element.get('text', ''))
    x = float(element.get('x', 0))
    y = float(element.get('y', 0))
    size = float(element.get('size', 1.0))
    align = str(element.get('align', 'bottom-left')).lower()
    rot_str = str(element.get('rot', 'R0')).upper()

    # Handle mirroring and rotation
    is_mirrored = False
    rot_angle_deg = 0.0
    if rot_str.startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_str[2:]) if len(rot_str) > 2 else 0.0
    elif rot_str.startswith('R'):
        rot_angle_deg = float(rot_str[1:]) if len(rot_str) > 1 else 0.0

    # Split text into lines
    lines = text.splitlines()
    num_lines = len(lines)
    max_line_len = max((len(line) for line in lines), default=0)

    # Estimate dimensions
    fontsize = size * 10
    text_width = int(max_line_len) * fontsize * 0.6 / 10.0
    text_height = int(num_lines) * fontsize * 1.2 / 10.0

    # Get polygon corners for the text box
    xs, ys = get_text_corners(
        x, y,
        align,
        rot_angle_deg,
        is_mirrored,
        text_width,
        text_height,
    )

    return min(xs), max(xs), min(ys), max(ys)

def get_list_of_sheet_text_bounding_boxes(schematic_info, space=1, ax=None):
    """
    Return a list of bounding box dictionaries for each sheet text element in the schematic.
    Each dictionary contains center coordinates, length, width, and metadata.
    Optionally draws boxes on the provided axis.
    """
    sheetinfo_texts = schematic_info['SheetInfo'].get('text', [])
    if not isinstance(sheetinfo_texts, list):
        sheetinfo_texts = [sheetinfo_texts]

    list_of_bounding_boxes = []

    for element in sheetinfo_texts:
        bbox = get_one_sheet_text_bounding_box(element)
        if bbox is None:
            continue
        element_min_x, element_max_x, element_min_y, element_max_y = bbox

        x_center = (element_min_x + element_max_x) / 2
        y_center = (element_min_y + element_max_y) / 2
        length = element_max_x - element_min_x
        width = element_max_y - element_min_y

        # Draw the bounding box (optional)
        # if ax is not None:
        #     draw_one_box(x_center, y_center, length + 1, width + 1, ax, color="#EEEA0A", thickness=2)

        bbox_info = {
            'part_name': 'SheetInfo',
            'gate_instance': element.get('text', ''),
            'x_center': x_center,
            'y_center': y_center,
            'length': length + 2 * space,
            'width': width + 2 * space,
            'key': 'text'
        }

        list_of_bounding_boxes.append(bbox_info)


    return list_of_bounding_boxes

def get_list_of_sheet_wire_bounding_boxes(schematic_info, space=1.0,ax=None):
    """
    Compute bounding boxes for each sheet wire segment.

    Parameters
    ----------
    sheet_wires : list or dict
        A list of wire segments (or a single wire dict) from the sheet.
    margin : float
        Padding around each bounding box.

    Returns
    -------
    list of dict
        Each dict contains bounding box info:
        {
            'type': 'sheet_wire',
            'x_center': ...,
            'y_center': ...,
            'length': ...,
            'width': ...,
            'xlim': (...),
            'ylim': (...)
        }
    """
    sheetinfo_wires = schematic_info['SheetInfo'].get('wire', [])
    if not isinstance(sheetinfo_wires, list):
        sheetinfo_wires = [sheetinfo_wires]

    list_of_bounding_boxes = []

    for wire in sheetinfo_wires:
        x1 = float(wire['x1'])
        y1 = float(wire['y1'])
        x2 = float(wire['x2'])
        y2 = float(wire['y2'])

        min_x = min(x1, x2) - space
        max_x = max(x1, x2) + space
        min_y = min(y1, y2) - space
        max_y = max(y1, y2) + space

        bbox = {
            'part_name': 'sheet_wire',
            'type': 'sheet_wire',
            'x_center': (min_x + max_x) / 2,
            'y_center': (min_y + max_y) / 2,
            'length': max_x - min_x,
            'width': max_y - min_y,
            'xlim': (min_x, max_x),
            'ylim': (min_y, max_y),
            'key': 'wire'
        }

        list_of_bounding_boxes.append(bbox)

    return list_of_bounding_boxes


### Bounding box Net

In [13]:
def get_bounding_boxes_for_net_elements(net_data,net_name, space=1.0,ax=None):
    """
    Returns a list of bounding boxes for wires, junctions, and labels in a net.

    Parameters
    ----------
    data : list
        List of segments/items (from net['segment']).
    net_name : str
        The name of the net (used for metadata).
    margin : float
        Margin to pad around each bounding box.

    Returns
    -------
    list of dict
        Each dict contains:
        {
            'type': 'wire' | 'junction' | 'label',
            'net_name': net_name,
            'x_center': ...,
            'y_center': ...,
            'length': ...,
            'width': ...,
            'xlim': (...),
            'ylim': (...)
        }
    """

    bbox_list = []

    for item in net_data:
        # ------------------- WIRE -------------------
        if 'wire' in item:
            wires = item['wire'] if isinstance(item['wire'], list) else [item['wire']]
            for wire in wires:
                # print("Processing wire:", wire)
                x1, y1 = float(wire['x1']), float(wire['y1'])
                x2, y2 = float(wire['x2']), float(wire['y2'])

                min_x = min(x1, x2) - space
                max_x = max(x1, x2) + space
                min_y = min(y1, y2) - space
                max_y = max(y1, y2) + space

                 
                bbox ={
                    'part_name': 'wire',
                    'type': 'wire',
                    'net_name': net_name,
                    'x_center': (min_x + max_x) / 2,
                    'y_center': (min_y + max_y) / 2,
                    'length': max_x - min_x,
                    'width': max_y - min_y,
                    'xlim': (min_x, max_x),
                    'ylim': (min_y, max_y),
                    'key': 'wire'
                }
                bbox_list.append(bbox)

                # # Optional: draw the wire and its bounding box
                # if ax is not None:
                #     ax.plot([x1, x2], [y1, y2], linewidth=2, color='green')
                #     draw_one_box(bbox['x_center'], bbox['y_center'],
                #                 bbox['length'], bbox['width'], ax=ax,
                #                 color="#00E5FF", thickness=2)
    
        # ------------------- JUNCTION -------------------
        if 'junction' in item:
            junctions = item['junction'] if isinstance(item['junction'], list) else [item['junction']]
            for junction in junctions:
                x, y = float(junction['x']), float(junction['y'])

                bbox ={
                    'part_name': 'junction',
                    'type': 'junction',
                    'net_name': net_name,
                    'x_center': x,
                    'y_center': y,
                    'length': space * 2,
                    'width': space * 2,
                    'xlim': (x - space, x + space),
                    'ylim': (y - space, y + space),
                    'key': 'junction'
                }

                bbox_list.append(bbox)

                # # Optional: draw the wire and its bounding box
                # if ax is not None:
                #     ax.plot([x1, x2], [y1, y2], linewidth=2, color='green')
                #     draw_one_box(bbox['x_center'], bbox['y_center'],
                #                 bbox['length'], bbox['width'], ax=ax,
                #                 color="#00E5FF", thickness=2)


        # ------------------- LABEL -------------------
        if 'label' in item:
            labels = item['label'] if isinstance(item['label'], list) else [item['label']]
            for label in labels:
                x = float(label['x'])
                y = float(label['y'])

                bbox ={
                    'part_name': 'label',
                    'type': 'label',
                    'net_name': net_name,
                    'x_center': x,
                    'y_center': y,
                    'length': space * 2,
                    'width': space * 2,
                    'xlim': (x - space, x + space),
                    'ylim': (y - space, y + space),
                    'key':'labels'
                }

                bbox_list.append(bbox)

                # Optional: draw the wire and its bounding box
                # if ax is not None:
                #     ax.plot([x1, x2], [y1, y2], linewidth=2, color='green')
                #     draw_one_box(bbox['x_center'], bbox['y_center'],
                #                 bbox['length'], bbox['width'], ax=ax,
                #                 color="#00E5FF", thickness=2)


    return bbox_list


### Draw Component

In [14]:
def draw_polygons(polygons, x=0, y=0, rot_angle_rad='R0',
                  color='red', fill=False, facecolor=None, ax=None):
    """
    Draw polygons (e.g., triangles in EAGLE symbols) with rotation and mirroring.

    Parameters
    ----------
    polygons : list
        Each dict has 'width' and a list of 'vertex' dicts with x, y.
    x, y : float
        Anchor position for the symbol instance.
    rot_angle_rad : str
        Rotation string like 'R0', 'R90', 'MR0', 'MR90'.
    color : str
        Outline color.
    fill : bool
        If True, fill the polygon.
    facecolor : str
        Color for fill (used if fill=True).
    ax : matplotlib axis
        Axis to draw on.
    """
    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    theta = math.radians(rot_angle_deg)

    cos_t, sin_t = math.cos(theta), math.sin(theta)

    if ax is None:
        ax = plt.gca()

    for poly in polygons:
        vertices = poly.get('vertex', [])
        rotated_pts = []

        for v in vertices:
            px = float(v['x'])
            py = float(v['y'])

            # Apply rotation
            rx = px * cos_t - py * sin_t
            ry = px * sin_t + py * cos_t

            # Apply mirroring across vertical axis
            if is_mirrored:
                rx = -rx

            # Translate
            rotated_x = x + rx
            rotated_y = y + ry

            rotated_pts.append((rotated_x, rotated_y))

        width = float(poly.get('width', '0.1524'))

        poly_patch = Polygon(
            rotated_pts,
            closed=True,
            fill=fill,
            facecolor=facecolor if (fill and facecolor is not None) else 'none',
            edgecolor=color,
            linewidth=width
        )
        ax.add_patch(poly_patch)


def draw_circle(circle_data, x=0, y=0, rot_angle_rad='R0', ax = None):
    """
    Draws a circle with rotation.
    """
    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    elif rot_angle_rad.upper().startswith('MR'):
        rot_angle_deg = float(rot_angle_rad[2:])
        is_mirrored = True 
    rot_angle_rad = math.radians(rot_angle_deg)
    
    # Extract circle parameters
    x_center = float(circle_data['x'])
    y_center = float(circle_data['y'])
    radius   = float(circle_data['radius'])
    line_w   = float(circle_data['width'])

    # Apply rotation to the circle center
    rotated_x_center = x + (x_center * math.cos(rot_angle_rad) - y_center * math.sin(rot_angle_rad))
    if is_mirrored:
        rotated_x_center = x + (x_center * math.cos(rot_angle_rad) + y_center * math.sin(rot_angle_rad))
    else:
        rotated_x_center = x + (x_center * math.cos(rot_angle_rad) - y_center * math.sin(rot_angle_rad))

    rotated_y_center = y + (x_center * math.sin(rot_angle_rad) + y_center * math.cos(rot_angle_rad))


    # Create a Circle patch (unfilled by default)
    circle_patch = Circle(
        (rotated_x_center, rotated_y_center),
        radius,
        fill=False,         
        linewidth=line_w*10,
        edgecolor='red'     
    )

    # Add to the current axes
    if ax is None:
        ax = plt.gca()
    ax.add_patch(circle_patch)


def compress_pad_string(pad_str):
    """
    Given a space-separated pad string like '3 4 5',
    returns a compressed format like '3*3'
    """
    parts = pad_str.strip().split()
    
    if not parts:
        return ""
    

    first = parts[0]
    count = len(parts)
    if count == 1:
        return first
    return f"{first}*{count}"


def draw_pins(pins, x=0, y=0, rot_angle_rad='R0', ax=None, schematic_info = None, lib_name = None, de_set_name = None, device_name = None, scale = 1.0, gate = None):
    """
    Draws pins with rotation.
    """
    if ax is None:
        ax = plt.gca()
    is_mirrored = False
    rot_angle_deg = 0.0
    if rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    elif rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    rot_angle_rad = math.radians(rot_angle_deg)
    # font_size = 12.0/100.0 * float(scale)

    # if is_mirrored:
    #     print("is_mirrored:", is_mirrored, "rot_angle_rad:", rot_angle_rad, "rot_angle_deg:", rot_angle_deg)
    # Define the mapping from length strings to extension distances
    length_map = {
        'point': 0.0,
        'short': 2.54,
        'middle': 2.54 * 2,
        'long': 2.54 * 3
    }

    rotated_pin_positions = []
    for pin in pins:
        x_base = float(pin['x'])
        y_base = float(pin['y'])
        x_rot = x + (x_base * math.cos(rot_angle_rad) - y_base * math.sin(rot_angle_rad))
        y_rot = y + (x_base * math.sin(rot_angle_rad) + y_base * math.cos(rot_angle_rad))
        if is_mirrored:
            x_rot = 2*x - x_rot
        rotated_pin_positions.append((x_rot, y_rot))

    def compute_min_spacing(pin_positions):
        min_dist = float('inf')
        for (x1, y1), (x2, y2) in itertools.combinations(pin_positions, 2):
            dx = x2 - x1
            dy = y2 - y1
            dist = math.hypot(dx, dy)
            if dist > 1e-6:
                min_dist = min(min_dist, dist)
        return min_dist
    min_spacing = compute_min_spacing(rotated_pin_positions)
    font_size = max(min(min_spacing * 100, 12), 10)


    for pin in pins:
        # print("pin name:", pin['name'])
        # Get base coordinates
        x_base = float(pin['x'])
        # if is_mirrored:
        #     x_base = -x_base
        y_base = float(pin['y'])

        dotted = pin.get('function','') == 'dot'

        # Apply rotation to the pin base
        rotated_x_base = x + (x_base * math.cos(rot_angle_rad) - y_base * math.sin(rot_angle_rad))
        rotated_y_base = y + (x_base * math.sin(rot_angle_rad) + y_base * math.cos(rot_angle_rad))

        # Determine extension length
        length_type = pin.get('length', 'long').lower()
        extension = length_map.get(length_type, 2.54)

        # Compute pin angle
        rot_pin_str = pin.get('rot', 'R0')
        pin_angle_deg = float(rot_pin_str[1:]) if rot_pin_str.upper().startswith('R') else (float(rot_pin_str[2:]))
        pin_angle_rad = math.radians(pin_angle_deg) + rot_angle_rad
        x_end = rotated_x_base + extension * math.cos(pin_angle_rad)
        y_end = rotated_y_base + extension * math.sin(pin_angle_rad)
        # Compute end of pin


        
        if is_mirrored and dotted:
            circle_centerx = rotated_x_base + (extension - 1) * math.cos(pin_angle_rad)
            circle_centery = rotated_y_base + (extension - 1) * math.sin(pin_angle_rad)
            x_end = rotated_x_base + (extension - 2) * math.cos(pin_angle_rad)
            y_end = rotated_y_base + (extension - 2) * math.sin(pin_angle_rad)

            x_end = 2*x - x_end
            rotated_x_base = 2*x - rotated_x_base
            #print("pin name:", pin['name'], "pin not_mirrored:", pin['name'], "rotated_x_base:", rotated_x_base, "rotated_y_base:", rotated_y_base, "x_end:", x_end, "y_end:", y_end, "calculated angle:", pin_angle_rad)
        elif is_mirrored:
            # x_end = rotated_x_base - extension * math.cos(pin_angle_rad)
            x_end = 2*x - x_end
            rotated_x_base = 2*x - rotated_x_base
            #print("pin name:", pin['name'], "pin is_mirrored:", pin['name'], "rotated_x_base:", rotated_x_base, "rotated_y_base:", rotated_y_base, "x_end:", x_end, "y_end:", y_end, "calculated angle:", pin_angle_rad)
        elif dotted:
            circle_centerx = rotated_x_base + (extension - 1) * math.cos(pin_angle_rad)
            circle_centery = rotated_y_base + (extension - 1) * math.sin(pin_angle_rad)
            x_end = rotated_x_base + (extension - 2) * math.cos(pin_angle_rad)
            y_end = rotated_y_base + (extension - 2) * math.sin(pin_angle_rad)
        
            #print("pin name:", pin['name'], "pin not_mirrored:", pin['name'], "rotated_x_base:", rotated_x_base, "rotated_y_base:", rotated_y_base, "x_end:", x_end, "y_end:", y_end, "calculated angle:", pin_angle_rad)
        
        dx = x_end - rotated_x_base
        dy = y_end - rotated_y_base
        length = math.hypot(dx, dy)
        if length > 1e-6:
            ux, uy = dx/length, dy/length
        else:
            ux, uy = 1.0, 0.0
    # change
        if length_type == 'point':
            offset_pts = -8
        elif length_type == 'short':
            offset_pts = -47
        elif length_type == 'middle':
            offset_pts = -75
        elif length_type == 'long':   
            offset_pts = -103
        offset_x = -ux * offset_pts
        # if is_mirrored:
        #     offset_x = -offset_x
        offset_y = -uy * (offset_pts - 8)
        pin_angle_deg_final = math.degrees(pin_angle_rad)
        pin_angle_deg_final = pin_angle_deg_final % 360
        align_map = {
            0:     ('left',  'center'),
            90:    ('left', 'center'),
            180:   ('right', 'center'),
            270:   ('right',  'center'),
        }
        ha, va = align_map.get(pin_angle_deg_final, ('center', 'center'))

        if is_mirrored:
            ha = 'right' if ha == 'left' else 'left'
        angel_drawed = pin_angle_deg_final
        if pin_angle_deg_final == 180:
            angel_drawed = 0

        if pin_angle_deg_final == 270:
            angel_drawed = 90



        # Draw the pin line and base marker
        if dotted:
            theta = np.linspace(0, 2 * np.pi, 100)
            ax.plot(circle_centerx + np.cos(theta), circle_centery + np.sin(theta), label = 'circle', color='red', lw=1, clip_on=True)
            ax.plot([rotated_x_base, x_end], [rotated_y_base, y_end], color='red', lw=1, clip_on=True)
        else:
            ax.plot([rotated_x_base, x_end], [rotated_y_base, y_end], color='red', clip_on=False, lw=1, zorder=20)
        ax.scatter([rotated_x_base], [rotated_y_base], color='blue', zorder=5, clip_on=True)
        
        # print('pin Name:', pin['name'], 'visible:', pin.get('visible', ''))

        if pin.get('visible', '').lower() in ['pin', 'both', '']:
                # ax.annotate(
                #     pin['name'],
                #     (rotated_x_base, rotated_y_base),
                #     textcoords="offset points",
                #     xytext=(offset_x, offset_y),
                #     ha=ha, va=va,
                #     fontsize=font_size, 
                #     color = 'Gray',
                #     rotation=angel_drawed,
                #     rotation_mode='anchor',    
                #     clip_on=True
                # )
                pin_name = pin['name'].split('@', 1)[0]
                ax.annotate(
                    pin_name,
                    (rotated_x_base, rotated_y_base),
                    textcoords="offset points",
                    xytext=(offset_x, offset_y),
                    ha=ha, va=va,
                    fontsize=font_size, 
                    color='Gray',
                    rotation=angel_drawed,
                    rotation_mode='anchor',    
                    clip_on=True
                )
        if pin.get('visible', '').lower() in ['pad', 'both', '']:
            # print("Pad Drawn: pin name:", pin['name'], "pin_angle_deg_final:", pin_angle_deg_final, "ha:", ha, "va:", va)
            connects = find_connects(schematic_info, library_name=lib_name, deviceset_name=de_set_name, device_name=device_name)
            with open('connects.json', 'w') as f:
                json.dump(connects, f, indent=2)
            for connect in connects:
                if connect['pin'] == pin['name'] and (connect['gate'] == gate or connect['gate'] is None or connect['gate'] == '' or gate is None or gate == ''):
                    if ha == 'left':
                        ha = 'right'
                    elif ha == 'right':
                        ha = 'left'
                    if va == 'top':
                        va = 'bottom'
                    elif va == 'bottom':
                        va = 'top'
                    if ha == 'left' and pin_angle_deg_final !=90 and pin_angle_deg_final != 270:
                        # print("left 0 or 180 Here")
                        if length_type == 'point' or length_type == 'short':
                            offset_x = offset_x + 40    
                        elif length_type == 'middle' or length_type == 'long':
                            offset_x = offset_x + 50
                    elif ha == 'right' and pin_angle_deg_final == 0:
                        if length_type == 'point' or length_type == 'short':
                            offset_x = offset_x -40
                        elif length_type == 'middle' or length_type == 'long':
                            offset_x = offset_x -60
                    elif ha == 'right' and pin_angle_deg_final == 180:
                        if length_type == 'point' or length_type == 'short':
                            offset_x = offset_x -40
                        elif length_type == 'middle' or length_type == 'long':
                            offset_x = offset_x -60



                    
                    if pin_angle_deg_final == 90:
                        
                        if length_type == 'point' or length_type == 'short':
                            offset_y = offset_y -50
                        elif length_type == 'middle' or length_type == 'long':
                            offset_y = offset_y -60
                    if pin_angle_deg_final == 270:
                        if length_type == 'point' or length_type == 'short':
                            offset_y = offset_y + 40    
                        elif length_type == 'middle' or length_type == 'long':
                            offset_y = offset_y + 50
                        # print("270 de pin/pad:",ha, va)

                    # elif pin_angle_deg_final == 90:
                    #     offset_x = offset_x  -100
                    # elif pin_angle_deg_final == 270:
                    #     offset_x = offset_x + 5
                      

                    if angel_drawed == 90 :
                        va = 'bottom'
                    # elif pin_angle_deg_final == 270:
                    #     va = 'bottom'
                    # elif ha == 'center' and va == 'bottom':
                    #     offset_x = offset_x
                    # elif ha == 'center' and va == 'top':
                    #     offset_x = offset_x +20

                    if va == 'center':
                        offset_y = offset_y -10
                    angel_drawed = pin_angle_deg_final
                    if pin_angle_deg_final == 180:
                        angel_drawed = 0

                    if pin_angle_deg_final == 270:
                        angel_drawed = 90
                    draw_text = compress_pad_string(connect['pad'])
                    if dotted and pin_angle_deg_final == 180:
                        
                        ax.annotate(
                            draw_text,
                            (x_end - 2, y_end),
                            textcoords="offset points",
                            xytext=(-offset_x, -offset_y),
                            ha=ha, va=va,
                            fontsize=font_size, 
                            color = 'Gray',
                            rotation=angel_drawed,            # ← rotate text
                            rotation_mode='anchor',    
                            clip_on=True
                        )
                    elif dotted and pin_angle_deg_final == 0:
                        ax.annotate(
                            draw_text,
                            (x_end + 1, y_end),
                            textcoords="offset points",
                            xytext=(-offset_x, -offset_y),
                            ha=ha, va=va,
                            fontsize=font_size, 
                            color = 'Gray',
                            rotation=angel_drawed,            # ← rotate text
                            rotation_mode='anchor',    
                            clip_on=True
                        )
                    else:
                        ax.annotate(
                            draw_text,
                            (x_end, y_end),
                            textcoords="offset points",
                            xytext=(-offset_x, -offset_y),
                            ha=ha, va=va,
                            fontsize=font_size, 
                            color = 'Gray',
                            rotation=angel_drawed,            # ← rotate text
                            rotation_mode='anchor',    
                            clip_on=True
                        )
 

    return rot_angle_rad


def draw_rectangle(rect_data, x=0, y=0, rot_angle_rad='R0', color='red', alpha=1.0, ax=None):
    """
    Draw a rectangle that supports rotation and mirroring (Rθ / MRθ).
    The rectangle is defined by two opposite corners (x1,y1)-(x2,y2) in local coords.
    Transform order: mirror -> rotate -> translate (about origin x,y).
    """
    if ax is None:
        ax = plt.gca()

    # Parse rotation/mirror
    rot_str = str(rot_angle_rad).upper()
    mirrored = False
    rot_deg = 0.0
    if rot_str.startswith('MR'):
        mirrored = True
        rot_deg = float(rot_str[2:] or 0.0)
    elif rot_str.startswith('R'):
        rot_deg = float(rot_str[1:] or 0.0)

    theta = math.radians(rot_deg)
    c, s = math.cos(theta), math.sin(theta)

    # Local rect corners (axis-aligned in local space)
    x1 = float(rect_data['x1']); y1 = float(rect_data['y1'])
    x2 = float(rect_data['x2']); y2 = float(rect_data['y2'])
    corners = [(x1,y1), (x1,y2), (x2,y2), (x2,y1)]

    def transform(px, py):
        # mirror in local space
        if mirrored:
            px = -px
        # rotate about origin
        rx = px * c - py * s
        ry = px * s + py * c
        # translate to (x, y)
        return (x + rx, y + ry)

    world_pts = [transform(px, py) for (px, py) in corners]

    patch = Polygon(world_pts, closed=True, facecolor=color, edgecolor='none', alpha=alpha)
    ax.add_patch(patch)


def get_arrow_triangle(x, y, angle_deg = 0, size = 20, mirror=False):
    tri = np.array([
        [-size / 2, 0],            # Tip of arrow
        [0, size / 2],             # Left base
        [0, -size / 2],            # Right base
    ])

    # Rotate the triangle around (0,0)
    theta = np.radians(angle_deg)
    rot_matrix = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta),  np.cos(theta)]
    ])
    tri_rotated = tri @ rot_matrix.T

    # Mirror in local frame (flip x-values at local origin)
    if mirror:
        tri_rotated[:, 0] *= -1  # flip x

    # Now translate to (x, y)
    tri_translated = tri_rotated + np.array([x, y])
    return tri_translated



def draw_part_attributes(attributes, part_name="Unknown", part_value="Unknown", symbol_name="Unknown", ax = None, rot_angle=None, gate = None, symbol_name_print = None, is_uservalue = 'no', value_in_part = None):
    """
    Draws part attributes such as name and value on the plot.

    Args:
        attributes (list or dict): List or dictionary of attributes to draw.
        x (float): X-coordinate of the center.
        y (float): Y-coordinate of the center.
        rot_angle_rad (float): Rotation angle in radians.
        part_name (str): Name of the part.
        part_value (str): Value of the part.
        symbol_name (str): Name of the symbol.
    """

    if not attributes:  # Check if attributes is empty
        return
    if ax is None:
        ax = plt.gca()
    if isinstance(attributes, dict):  # Convert dictionary to list for uniform processing
        attributes = [attributes]

    for attribute in attributes:
        if attribute.get('display', 'on').lower() == 'off':
            continue

        attr_name = attribute.get('name', '')
        attr_x = float(attribute.get('x', 0))
        attr_y = float(attribute.get('y', 0))
        attr_size = float(attribute.get('size', 1.0))
        xref_yes  = attribute.get('xref', 'no').lower() == 'yes'
        attr_rot = attribute.get('rot', 'R0')
        attr_align = attribute.get('align', 'bottom-left').lower()
        attr_rot = attr_rot.upper()
        is_mirrored = attr_rot.startswith('MR')
        rot_val = attr_rot[2:] if is_mirrored else attr_rot[1:]
        rot_deg = int(rot_val)




        ha, va = get_text_alignment(attr_align, rot_deg, is_mirrored)
        
        attr_rot_deg = rot_deg


        if attr_rot_deg == 180:
            attr_rot_deg = 0
        elif attr_rot_deg == 270:
            attr_rot_deg = 90
        
        bbox_kw = None
        if xref_yes:
            bbox_kw = dict(
                boxstyle="square,pad=0.2",
                facecolor="white",
                edgecolor="gray",
                linewidth=1
            )

        # print("Drawing attribute:", attr_name, "at position:", attr_x, attr_y, "with rotation:", attr_rot_deg, "and size:", attr_size, 'ha', ha, 'va:', va)
        # Determine if the attribute is for part name or part value
        if attr_name == "NAME" and part_name != "Unknown":

            ax.text(attr_x, attr_y, part_name, fontsize= float(attr_size) * 10, ha=ha, va=va, 
                        rotation=attr_rot_deg, color='blue')
            
        elif attr_name == "VALUE":
            # Use symbol_name as part_value if part_value is unknown
            # print(f"Drawing attribute '{attr_name}' for part '{part_name}' with value '{part_value}'")
            display_value = part_value if part_value not in ["Unknown", ""] else None

            if 'GND' in part_name.upper():
                # Remove all digits but keep letters, dashes, etc.
                display_value = re.sub(r'\d+', '', part_name)
            
            if is_uservalue.lower() == 'no':
                ax.text(attr_x, attr_y, display_value, fontsize= float(attr_size) * 10, ha=ha, va=va, 
                        rotation=attr_rot_deg, color='green')
            elif is_uservalue.lower() == 'yes':
                if value_in_part != 'Unknown':
                    ax.text(attr_x, attr_y, value_in_part, fontsize= float(attr_size) * 10, ha=ha, va=va, 
                        rotation=attr_rot_deg, color='green')
            
        elif  "GND" != symbol_name:
            # print(f"Drawing attribute '{attr_name}' for part '{part_name}'")
            ax.text(attr_x, attr_y, attr_name, fontsize= float(attr_size) * 10, ha=ha, va=va, 
                        rotation=attr_rot_deg, color='red')
            

        cross_size = 0.3
        ax.plot([attr_x - cross_size, attr_x + cross_size],
        [attr_y, attr_y],
        color='lightgray', linewidth=1, zorder=0)


        ax.plot([attr_x, attr_x],
                [attr_y + cross_size, attr_y - cross_size],
                color='lightgray', linewidth=1, zorder=0)

     
def draw_texts(attributes, part_name="Unknown", part_value="Unknown", symbol_name="Unknown", ax = None, rot_angle_rad=None, gate = None, x = 0, y = 0):
    """
    Draws part attributes such as name and value on the plot.

    Args:
        attributes (list or dict): List or dictionary of attributes to draw.
        x (float): X-coordinate of the center.
        y (float): Y-coordinate of the center.
        rot_angle_rad (float): Rotation angle in radians.
        part_name (str): Name of the part.
        part_value (str): Value of the part.
        symbol_name (str): Name of the symbol.
    """
    # if part_name == "J6":
    #     import json
    #     with open("example_text_part.json", "w") as f:
    #                     json.dump(attributes, f, indent=4)
    if not attributes:  # Check if attributes is empty
        return
    is_mirrored = False
    rot_angle_deg = 0.0
    if rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    elif rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    rot_angle_rad = math.radians(rot_angle_deg)

    if ax is None:
        ax = plt.gca()
    if isinstance(attributes, dict):  # Convert dictionary to list for uniform processing
        attributes = [attributes]

    for attribute in attributes:
        if attribute.get('display', 'on').lower() == 'off':
            continue
        attr_name = attribute.get('text', '')
        if attr_name.startswith('>'):
            continue
        # print("symbol_text:",attribute)
        x_base = float(attribute.get('x', 0))
        y_base = float(attribute.get('y', 0))
        attr_size = float(attribute.get('size', 1.0))
        xref_yes  = attribute.get('xref', 'no').lower() == 'yes'
        attr_rot = attribute.get('rot', 'R0')
        attr_align = attribute.get('align', 'bottom-left').lower()

        pin_angle_deg = float(attr_rot[1:]) if attr_rot.upper().startswith('R') else (float(attr_rot[2:]))
        pin_angle_rad = math.radians(pin_angle_deg) + rot_angle_rad
        pin_angle_deg_final = math.degrees(pin_angle_rad)

        x_rot = x + (x_base * math.cos(rot_angle_rad) - y_base * math.sin(rot_angle_rad))
        y_rot = y + (x_base * math.sin(rot_angle_rad) + y_base * math.cos(rot_angle_rad))
        if is_mirrored:
            x_rot = 2*x - x_rot


        attr_x = x_rot
        attr_y = y_rot

        ha,va = get_text_alignment(attr_align, pin_angle_deg_final, is_mirrored)

        attr_rot_deg = pin_angle_deg_final


        if attr_rot_deg == 180:
            attr_rot_deg = 0
        elif attr_rot_deg == 270:
            attr_rot_deg = 90
        
        bbox_kw = None
        if xref_yes:
            bbox_kw = dict(
                boxstyle="square,pad=0.2",
                facecolor="white",
                edgecolor="gray",
                linewidth=1
            )

        # print("Drawing attribute:", attr_name, "at position:", attr_x, attr_y, "with rotation:", attr_rot_deg, "and size:", attr_size, 'ha', ha, 'va:', va)
        # Determine if the attribute is for part name or part value


        layer = attribute.get('layer', '-1')  # Default to layer '-1' if not specified

        color = 'black'
        if layer  in ['97','95','96']:
            color = 'gray'
        elif layer == '94':
            color = 'red'
        elif layer == '98':
            color = '#d7d769'

        ax.text(attr_x, attr_y, attr_name, fontsize=float(attr_size) * 10, ha=ha, va=va, 
                        rotation=attr_rot_deg, color=color)
            

        cross_size = 0.3
        ax.plot([attr_x - cross_size, attr_x + cross_size],
        [attr_y, attr_y],
        color='lightgray', linewidth=1, zorder=0)


        ax.plot([attr_x, attr_x],
                [attr_y + cross_size, attr_y - cross_size],
                color='lightgray', linewidth=1, zorder=0)


def draw_wire_eagle_style(wire, x=0, y=0, rot_angle_rad='R0', ax=None):
    """
    Draws a wire or arc with support for rotation and mirroring (EAGLE style).
    Fixes mirroring bug for arcs (curve sign inversion).
    """
    import math
    from matplotlib.patches import Arc
    import matplotlib.pyplot as plt
    # print("wire drawn: x: ", x,",y: ", y, "rot_angle_rad: ", rot_angle_rad)
    # Parse rotation & mirror
    rot_angle_deg = 0.0
    is_mirrored = False
    if rot_angle_rad.upper().startswith('MR'):
        is_mirrored = True
        rot_angle_deg = float(rot_angle_rad[2:])
    elif rot_angle_rad.upper().startswith('R'):
        rot_angle_deg = float(rot_angle_rad[1:])
    theta = math.radians(rot_angle_deg)

    # Rotation matrix
    cosθ, sinθ = math.cos(theta), math.sin(theta)
    rot_matrix = [[cosθ, -sinθ], [sinθ, cosθ]]

    # Mirror matrix
    mirror_matrix = [[-1, 0], [0, 1]] if is_mirrored else [[1, 0], [0, 1]]
    # mirror_matrix = [[1, 0], [0, 1]]
    def transform_point(px, py):
        # First apply mirroring
        if is_mirrored:
            px = -px
        # Then apply rotation
        rx = px * rot_matrix[0][0] + py * rot_matrix[0][1]
        ry = px * rot_matrix[1][0] + py * rot_matrix[1][1]
        return x + rx, y + ry

    def transform_point(px, py):
        # Apply rotation
        rx = px * rot_matrix[0][0] + py * rot_matrix[0][1]
        ry = px * rot_matrix[1][0] + py * rot_matrix[1][1]
        # Apply mirror
        mx = rx * mirror_matrix[0][0] + ry * mirror_matrix[0][1]
        my = rx * mirror_matrix[1][0] + ry * mirror_matrix[1][1]
        # Translate
        return x + mx, y + my
    
    # Extract and transform endpoints
    x1_orig, y1_orig = float(wire['x1']), float(wire['y1'])
    x2_orig, y2_orig = float(wire['x2']), float(wire['y2'])
    x1, y1 = transform_point(x1_orig, y1_orig)
    x2, y2 = transform_point(x2_orig, y2_orig)
    
    # if is_mirrored:
    #     x2, y2 = transform_point(x1_orig, y1_orig)
    #     x1, y1 = transform_point(x2_orig, y2_orig)

    # Get curve (adjust if mirrored)
    curve_deg = float(wire.get('curve', 0))
    if is_mirrored:
        curve_deg *= -1  # 🔁 key fix: mirror inverts curve direction

    if abs(curve_deg) < 1e-9:
        if ax is None: ax = plt.gca()
        ax.plot([x1, x2], [y1, y2], 'r-')
        return

    #  Arc geometry (based on original coords)
    chord_len = math.hypot(x2_orig - x1_orig, y2_orig - y1_orig)
    if chord_len < 1e-9:
        return

    theta_arc = math.radians(abs(curve_deg))
    R = chord_len / (2.0 * math.sin(theta_arc / 2.0))
    mx_orig = (x1_orig + x2_orig) / 2.0
    my_orig = (y1_orig + y2_orig) / 2.0
    d = math.sqrt(R * R - (chord_len / 2.0) ** 2)

    vx, vy = x2_orig - x1_orig, y2_orig - y1_orig
    nx, ny = -vy, vx
    length_n = math.hypot(nx, ny)
    nx, ny = nx / length_n, ny / length_n
    if not is_mirrored:
        if curve_deg > 0:
            cx_orig = mx_orig + d * nx
            cy_orig = my_orig + d * ny
        else:
            cx_orig = mx_orig - d * nx
            cy_orig = my_orig - d * ny
    else:
        if curve_deg > 0:
            cx_orig = mx_orig - d * nx
            cy_orig = my_orig - d * ny
        else:
            cx_orig = mx_orig + d * nx
            cy_orig = my_orig + d * ny

    # Transform arc center
    cx, cy = transform_point(cx_orig, cy_orig)

    # Compute angles (in transformed space)
    start_angle = math.degrees(math.atan2(y1 - cy, x1 - cx)) % 360
    end_angle = math.degrees(math.atan2(y2 - cy, x2 - cx)) % 360

    if curve_deg > 0:
        if end_angle < start_angle:
            end_angle = end_angle + 360
    else:
        if start_angle < end_angle:
            start_angle = start_angle + 360
        start_angle, end_angle = end_angle, start_angle

    # Draw arc
    linewidth = float(wire.get('width', 0.1524))
    if ax is None: ax = plt.gca()
    arc = Arc((cx, cy), 2 * R, 2 * R, angle=0,
              theta1=start_angle, theta2=end_angle,
              color='red', linewidth=linewidth * 10)
    ax.add_patch(arc)
    # ax.plot(cx, cy, 'go', markersize=8)
    # ax.plot(x1, y1, 'go', markersize=1)
    # ax.plot(x2, y2, 'go', markersize=1)
    # ax.plot([x1, x2], [y1, y2], 'r-', linewidth=linewidth * 10)
    

In [15]:
def draw_rectangle(rect_data, x=0, y=0, rot_angle_rad='R0', color='red', alpha=1.0, ax=None):
    """
    Draw a rectangle that supports rotation and mirroring (Rθ / MRθ).
    The rectangle is defined by two opposite corners (x1,y1)-(x2,y2) in local coords.
    Transform order (EAGLE-style): rotate -> mirror_in_world_Y -> translate.
    """
    if ax is None:
        ax = plt.gca()

    # --- parse rotation / mirror ---
    rot_str = str(rot_angle_rad).upper()
    mirrored = rot_str.startswith('MR')
    rot_deg = float(rot_str[2:] if mirrored else rot_str[1:] or 0.0)

    # CCW math rotation; EAGLE stores θ as a value, not a direction change.
    theta = math.radians(rot_deg)
    c, s = math.cos(theta), math.sin(theta)

    # local axis-aligned corners
    x1 = float(rect_data['x1']); y1 = float(rect_data['y1'])
    x2 = float(rect_data['x2']); y2 = float(rect_data['y2'])
    corners = [(x1, y1), (x1, y2), (x2, y2), (x2, y1)]

    def transform(px, py):
        # 1) rotate in local space (CCW)
        rx = px * c - py * s
        ry = px * s + py * c

        # 2) mirror in WORLD Y (flip x after rotation)
        if mirrored:
            rx = -rx

        # 3) translate to (x, y)
        return (x + rx, y + ry)

    world_pts = [transform(px, py) for (px, py) in corners]

    patch = Polygon(world_pts, closed=True, facecolor=color, edgecolor='none', alpha=alpha)
    ax.add_patch(patch)

### Draw Symbol

In [16]:
def visualize_wires_junctions(data, net_name, ax=None):
    """
    Visualizes wires, junctions, and labels from the given data using matplotlib.
    """
    if ax is None:
        ax = plt.gca()
    ax.yaxis.set_inverted(True)
    ax.set_aspect('equal', adjustable='box')

    def plot_wire(wire_seg):
        x1, y1 = float(wire_seg['x1']), float(wire_seg['y1'])
        x2, y2 = float(wire_seg['x2']), float(wire_seg['y2'])
        width = float(wire_seg.get('width', 0.1524))
        ax.plot([x1, x2], [y1, y2], linewidth=width * 10, color='green')

    def plot_junction(junction):
        x, y = float(junction['x']), float(junction['y'])
        ax.plot(x, y, marker='o', markersize=5, color='green')

    def plot_label(label):
        x, y = float(label['x']), float(label['y'])
        font_size = float(label.get('size', 8)) * 15
        rot_str = label.get('rot', 'R0').upper()

        is_mirrored = rot_str.startswith('MR')
        rot = int(rot_str[2:] if is_mirrored else rot_str[1:])
        if is_mirrored:
            rot = 360 - rot

        is_xref = label.get('xref', '').lower() == 'yes'

        # Horizontal alignment and offset
        if not is_mirrored:
            ha = 'left' if rot in (0, 90) else 'right' if rot in (180, 270) else 'center'
        else:
            ha = 'right' if rot in (0, 90) else 'left' if rot in (180, 270) else 'center'

        # Rotation adjustments for display
        disp_rot = 0 if rot == 180 else 90 if rot == 270 else rot
        va = 'center'
        offset_x, offset_y = (0, 0) if rot in (90, 270) else (1 if ha == 'left' else -1 if ha == 'right' else 0, 0)

        bbox_kw = dict(
            boxstyle="square,pad=0.2",
            facecolor="white",
            edgecolor="gray",
            linewidth=1
        ) if is_xref else None

        tri = get_arrow_triangle(x + offset_x, y + offset_y, rot, 2.5, is_mirrored)
        ax.text(
            x + offset_x, y + offset_y, net_name,
            fontsize=font_size, ha=ha, va=va, color='gray',
            bbox=bbox_kw, clip_on=True,
            rotation=disp_rot, rotation_mode='anchor'
        )

        if is_xref:
            ax.add_patch(Polygon(tri, closed=True, color='gray'))

    for item in data:
        if 'wire' in item:
            wires = item['wire'] if isinstance(item['wire'], list) else [item['wire']]
            for w in wires:
                plot_wire(w)

        if 'junction' in item:
            junctions = item['junction'] if isinstance(item['junction'], list) else [item['junction']]
            for j in junctions:
                plot_junction(j)

        if 'label' in item:
            labels = item['label'] if isinstance(item['label'], list) else [item['label']]
            for lbl in labels:
                plot_label(lbl)


def visualize_bus(data, net_name, ax=None):
    """
    Visualizes wires, junctions, and labels from the given data using matplotlib.
    """
    if ax is None:
        ax = plt.gca()
    ax.yaxis.set_inverted(True)
    ax.set_aspect('equal', adjustable='box')

    def plot_wire(wire_seg):
        x1, y1 = float(wire_seg['x1']), float(wire_seg['y1'])
        x2, y2 = float(wire_seg['x2']), float(wire_seg['y2'])
        width = float(wire_seg.get('width', 0.1524))
        ax.plot(
            [x1, x2], [y1, y2],
            linewidth=width * 15,
            color='#38ABDF',
            alpha=0.8  # 0 = fully transparent, 1 = fully opaque
        )


    def plot_label(label):
        x, y = float(label['x']), float(label['y'])
        font_size = float(label.get('size', 8)) * 15
        rot_str = label.get('rot', 'R0').upper()

        is_mirrored = rot_str.startswith('MR')
        rot = int(rot_str[2:] if is_mirrored else rot_str[1:])
        if is_mirrored:
            rot = 360 - rot

        is_xref = label.get('xref', '').lower() == 'yes'

        # Horizontal alignment and offset
        if not is_mirrored:
            ha = 'left' if rot in (0, 90) else 'right' if rot in (180, 270) else 'center'
        else:
            ha = 'right' if rot in (0, 90) else 'left' if rot in (180, 270) else 'center'

        # Rotation adjustments for display
        disp_rot = 0 if rot == 180 else 90 if rot == 270 else rot
        va = 'center'
        offset_x, offset_y = (0, 0) if rot in (90, 270) else (1 if ha == 'left' else -1 if ha == 'right' else 0, 0)

        bbox_kw = dict(
            boxstyle="square,pad=0.2",
            facecolor="white",
            edgecolor="gray",
            linewidth=1
        ) if is_xref else None

        tri = get_arrow_triangle(x + offset_x, y + offset_y, rot, 2.5, is_mirrored)
        ax.text(
            x + offset_x, y + offset_y, net_name,
            fontsize=font_size, ha=ha, va=va, color='gray',
            bbox=bbox_kw, clip_on=True,
            rotation=disp_rot, rotation_mode='anchor'
        )

        if is_xref:
            ax.add_patch(Polygon(tri, closed=True, color='gray'))

    
    for element_name,element_data in data.items():
        if 'wire' in element_name:
            wires = element_data if isinstance(element_data, list) else [element_data]
            for w in wires:
                plot_wire(w)

        if 'label' in element_name:
            labels = element_data if isinstance(element_data, list) else [element_data]
            for lbl in labels:
                plot_label(lbl)


def visualize_symbol_from_dict(symbol_data, x=0, y=0, rot_angle_rad='R0', part_name="Unknown",display_part_name='Unknown', part_value="Unknown",symbol_name="Unknown", ax = None, gate = None, schematic_info  = None, lib_name = None, de_set_name = None, device_name = None, symbol_name_print = None, is_uservalue = 'no', value_in_part = None):
    """
    Visualizes the symbol data with rotation.

    Args:
        symbol_data (dict): Dictionary containing symbol information.
        x (float): X-coordinate of the center.
        y (float): Y-coordinate of the center.
        rot (str): Rotation of the symbol, e.g., 'R90', 'R0', 'R270'.
    """
    # Convert rotation string to radians
    # rot_angle_deg = 0.0
    # if rot.upper().startswith('R'):
    #     rot_angle_deg = float(rot[1:])
    # rot_angle_rad = math.radians(rot_angle_rad)

    # Extract symbol elements
    pins = symbol_data.get('pin', [])
    # print("pins:",pins)
    wires = symbol_data.get('wire', [])
    rectangles = symbol_data.get('rectangle', [])
    circles = symbol_data.get('circle', [])
    polygons = symbol_data.get('polygon', [])
    attributes = symbol_data.get('attribute', [])
    texts = symbol_data.get('symbol_text', [])
    # print("here is the texts:",texts)
    # if part_name == "J6":
    #     import json
    #     with open("example_text_symbol.json", "w") as f:
    #                     json.dump(symbol_data, f, indent=4)
    if gate == "G$1":
        gate = None


    # Plot circles
    for circle in circles:
        
        draw_circle(circle, x, y, rot_angle_rad, ax)


    # Plot polygons
    if polygons:
        
        draw_polygons(polygons, x, y, rot_angle_rad, color='red', fill=True, facecolor='red', ax = ax)


    # Plot pins
    # print("start to draw pins")
    # print("pins")
    
    angle = draw_pins(pins, x, y, rot_angle_rad, ax, schematic_info = schematic_info, lib_name = lib_name, de_set_name = de_set_name, device_name = device_name, gate = gate)
    
    # Draw wires
    for wire in wires:
        draw_wire_eagle_style(wire, x, y, rot_angle_rad, ax)

                                                                
    # Draw rectangles

    for rect_data in rectangles:
        
        draw_rectangle(rect_data, x, y, rot_angle_rad, color='red', alpha=1.0, ax =ax)


    # print("here are the texts:",texts)
    draw_texts(texts, part_name, part_value, symbol_name, ax = ax, rot_angle_rad=rot_angle_rad, gate = gate, x = x, y = y)
    

    draw_part_attributes(attributes, display_part_name, part_value, symbol_name, ax = ax, rot_angle=angle, gate = gate, symbol_name_print =symbol_name_print,  is_uservalue = is_uservalue, value_in_part = value_in_part)

    # Draw cross
    cross_size = 0.5
    ax.plot([x - cross_size, x + cross_size], [y, y],
            color="#ff6666", linewidth=1, zorder=0)
    ax.plot([x, x], [y + cross_size, y - cross_size],
            color="#ff6666", linewidth=1, zorder=0)

    

In [17]:
# Function to visualize symbol with rotation
from traceback import print_tb



def visualize_schematic(schematic_info, save_path=None,draw_grid=True,draw_bounding_bbox=False):
    """
    Visualizes the schematic based on the provided schematic_info.

    Args:
        schematic_info (dict): Dictionary containing schematic information, including parts and nets.
    """
    
    list_of_bounding_box = []

    list_of_symbol_bounding_box, element_lists = get_list_of_symbol_bounding_boxes(schematic_info,space=1)

    list_of_sheet_text_bounding_box = get_list_of_sheet_text_bounding_boxes(schematic_info,space=1)

    list_of_sheet_wire_bounding_box = get_list_of_sheet_wire_bounding_boxes(schematic_info, space=1)

    list_of_net_bounding_box = []
    bounding_boxes = []

    for net_name, net_data in schematic_info['nets'].items():
        list_of_net_bounding_box.extend(
            get_bounding_boxes_for_net_elements(net_data, net_name, space=0.5)
        )

    bounding_boxes.extend(list_of_sheet_text_bounding_box)
    bounding_boxes.extend(list_of_symbol_bounding_box)
    bounding_boxes.extend(list_of_sheet_wire_bounding_box)
    bounding_boxes.extend(list_of_net_bounding_box)
    bounding_box_full = get_bounding_box_full_sch(bounding_boxes)[0]

    if not bounding_box_full:
        print("Empty schematic: ", schematic_info)
        return None
         


    list_of_bounding_box.extend(list_of_sheet_text_bounding_box)
    list_of_bounding_box.extend(list_of_symbol_bounding_box)
    list_of_bounding_box.extend(list_of_sheet_wire_bounding_box)
    list_of_bounding_box.extend(list_of_net_bounding_box)
    # list_of_bounding_box.extend([bounding_box_full])

    x_center = bounding_box_full['x_center']
    y_center = bounding_box_full['y_center']
    sch_length = bounding_box_full['length']
    sch_width =  bounding_box_full['width']
    xlim = [x_center - sch_length/2, x_center + sch_length/2]
    ylim = [y_center - sch_width/2, y_center + sch_width/2]
    # (xmin, xmax), (ymin, ymax) = bounding_box
    raw_w  = sch_length
    raw_h  = sch_width
    # print("schematics_raw_dimensions:", raw_w, raw_h)
    fig_w    = raw_w  / 5.0
    fig_h    = raw_h  / 5.0

    
    # print("Dimensions of schematic:", x, y)
    # if (x == -1):
    #     print("Error: Invalid schematic file or empty schematic.")
    #     return
    fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=100)  # Create figure and axes
    # ax = plt.gca()  # Get current axes
    # print("Parts in schematic:", schematic_info['parts'])

      # Parse the schematic file
    import json
    with open("example_symbol_part.json", "w") as f:
                    json.dump(schematic_info, f, indent=4)
    # Visualize parts
    for part_name, part_data in schematic_info['parts'].items():
        part_value = part_data.get('value', 'Unknown')
        lib_name = part_data.get('library', 'Unknown')
        de_set_name = part_data.get('deviceset', 'Unknown')
        device_name = part_data.get('device', 'Unknown')
        symbol_name_print = part_data.get('name', 'Unknown')
        value_in_part = part_data.get('value_in_part','Unkown')
        

        for instance in part_data.get('instances', []):
            symbol_name = instance['symbol']
            x = float(instance['x'])
            y = float(instance['y'])
            rot = instance['rot']
            gate = instance['gate']
            display_part_name = part_name+gate if len(part_data.get('instances', []))>1 else part_name
            symbol_element = get_symbol_element_of_instance_from_library(
                schematic_info['IC_library'], part_name, gate, schematic_info['parts']
            )
            deviceset_element = get_deviceset_element(schematic_info['IC_library'], lib_name, de_set_name)
            is_uservalue = deviceset_element.get('uservalue', 'no')
            # print("part_data:", symbol_name)
            # test
            if symbol_element:
                 visualize_symbol_from_dict(symbol_element, x, y, rot, part_name, display_part_name,part_value, symbol_name, gate = gate, schematic_info = schematic_info, ax = ax, lib_name = lib_name, de_set_name = de_set_name, device_name = device_name, symbol_name_print = symbol_name_print, is_uservalue = is_uservalue, value_in_part = value_in_part)

    # Visualize nets
    for net_name, net_data in schematic_info['nets'].items():
        visualize_wires_junctions(net_data, net_name, ax=ax)


    # Visualize sheet
    sheetinfo_wires = schematic_info['SheetInfo'].get('wire', [])
    sheetinfo_texts = schematic_info['SheetInfo'].get('text', [])
    
    visualize_sheet_wires(sheetinfo_wires,ax=ax)
    visualize_sheet_texts(sheetinfo_texts,ax=ax)
    

    busses = schematic_info['bus']

    for bus in busses:
        bus_name = bus['name']
        bus_wire = bus['segment']
        visualize_bus(bus_wire, bus_name, ax=ax)

    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

    # Draw bounding box in red dashed line
    rect = Rectangle(
        (xlim[0], ylim[0]),  # Bottom-left corner
        xlim[1] - xlim[0],   # Width
        ylim[1] - ylim[0],   # Height
        linewidth=10,
        edgecolor='red',
        linestyle='--',
        facecolor='none'
    )
    ax.add_patch(rect)

    if draw_bounding_bbox:
        # draw_list_of_bounding_boxes(list_of_symbol_bounding_box,ax, color='orange')
        # draw_list_of_bounding_boxes(list_of_sheet_text_bounding_box,ax, color='gray')
        # draw_list_of_bounding_boxes(list_of_sheet_wire_bounding_box,ax, color='gray')
        # draw_list_of_bounding_boxes(list_of_net_bounding_box,ax, color='blue')

        # draw_list_of_bounding_boxes(list_of_bounding_box,ax, color='blue')

        # draw_element_list(element_lists, draw_list=[],ax =ax)
        pass


    setting = schematic_info['setting']
    visualize_grid_from_setting(setting,ax=ax,draw_grid=draw_grid)


    ax.set_xlabel(None)
    ax.set_ylabel(None)
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    if save_path:
        print("plt saved")
        plt.savefig(save_path, bbox_inches='tight')
        
    fig = plt.gcf()
    fig.canvas.draw()


    src_xlim = ax.get_xlim()
    src_ylim = ax.get_ylim()
    src_width = src_xlim[1] - src_xlim[0]
    src_height = src_ylim[1] - src_ylim[0]



    bbox_inches = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
    pixel_width = bbox_inches.width * fig.dpi
    pixel_height = bbox_inches.height * fig.dpi


    pixels_per_unit_x = pixel_width / src_width
    pixels_per_unit_y = pixel_height / src_height
    return pixels_per_unit_x, pixels_per_unit_y, src_width, src_height, pixel_width, pixel_height, list_of_bounding_box, xlim, ylim



In [18]:
def visualize_ful_schematic(schematic_file_path, save_path=None,draw_grid=True,draw_bounding_bbox=False):
    """
    Visualizes the schematic based on the provided schematic_info.

    Args:
        schematic_info (dict): Dictionary containing schematic information, including parts and nets.
    """
    list_of_schematic_info = process_schematic_file(schematic_file_path)
    
    for i, schematic_info in enumerate(list_of_schematic_info):
        new_path = save_path.replace(".png", f"#{i}.png")
        visualize_schematic(
            schematic_info,
            save_path=new_path,
            draw_grid=False,
            draw_bounding_bbox=True
        )


# Example

In [20]:
if __name__ == "__main__":
    # Load and parse the XML file for the specified IC
    # folder_path = r"/Users/linkaiyuan/文件/PSU/test_single_sch/A111_Pulsed_Radar_Breakout_SparkFun-Connectors_RASPBERRYPI-26-PIN-GPIO.sch"
    # # folder_path = r"F:\GitHub\PCBEDA\sample PCB\sparkfun\simplified_Module_library"
    # IC_name = "ICM-20948"
    # sch_file_name = f"{folder_path}\\{IC_name}\\{IC_name}.sch"
    # # sch_file_name = r"F:\GitHub\PCBEDA\sample PCB\custom\template.sch"
    # visualize_schematic(folder_path)


    output_dir = '/Users/taitinglu/Desktop/IMG2SCH/test1.png'
    # output_dir = r"C:\Users\Taiting\Desktop\yolo_training_24\data\ArtemisDevKit.png"
    # sch_file = "F:\GitHub\IMG2SCH\sample\ArtemisDevKit.sch" 
    sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\yolo_training_24\dataset\val\sch\MagJack_Breakout.sch"
    # visualize_schematic_tiled(sch_file, output_dir, rows=3, cols=3, dpi=300)
    # sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\yolo_training_24\dataset\val\sch\Breadboard Power Supply - 5-3.3SMD_V14.sch"
    sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\data\full sch\EasyDriver_-_Stepper_Motor_Driver.sch"
    # sch_file = r"C:\Users\Taiting\Desktop\yolo_training_24\data\Arduino_Nano_Every.sch"
    # sch_file = "/Users/linkaiyuan/Library/Containers/com.tencent.xinWeChat/Data/Documents/xwechat_files/wxid_cpkwgyls8hon32_7a57/msg/file/2025-08/LilyPad_E-Sewing_ProtoSnap.sch"
    # sch_file = "/Users/linkaiyuan/文件/PSU/full sch unzip/ARGOS_Satellite_Transceiver_Shield_-_ARTIC_R2.sch"
    sch_file = r"/Users/taitinglu/Desktop/IMG2SCH/Arduino_Nano_Every.sch"
    sch_file = '/Users/taitinglu/Desktop/IMG2SCH/AWS_IoT_ExpressLink_SARA-R5_Starter_Kit copy.sch'
    
    visualize_ful_schematic(sch_file, save_path=output_dir,draw_grid=False,draw_bounding_bbox=True)

plt saved
plt saved


# Preprocess eagle.zip

In [ ]:
import os
import shutil
import zipfile

def check_zip_in_subfolders(folder_path):
    """
    Checks if each subfolder inside folder_path contains a 'eagle.zip' file.
    Returns a list of subfolders that contain 'eagle.zip'.
    """
    result = []
    for subfolder in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder)
        if os.path.isdir(subfolder_path):
            expected_zip = os.path.join(subfolder_path, "eagle.zip")
            if os.path.isfile(expected_zip):
                result.append(subfolder)
    return result


def unzip_eagle_zips_in_subfolders(folder_path, clean_when_single_pair=True):
    """
    For each *directory* inside folder_path:
      - Unzip 'eagle.zip' if present
      - If no .sch in the subfolder after unzip, search one level deeper and move .sch/.brd up
      - Print helpful messages when .sch missing or only nested folders exist
      - If exactly one .sch and one .brd exist, optionally delete everything else
      - Rename .sch and .brd to match the subfolder's name (conflict-safe)

    Args:
        folder_path (str): Path containing many subfolders
        clean_when_single_pair (bool): If True, delete everything except the single .sch/.brd pair
    """

    if not os.path.isdir(folder_path):
        raise ValueError(f"Not a directory: {folder_path}")

    def is_hidden(name: str) -> bool:
        return name.startswith(".")

    for subfolder in os.listdir(folder_path):
        if is_hidden(subfolder):
            continue  # skip .DS_Store, .git, etc.

        subfolder_path = os.path.join(folder_path, subfolder)
        if not os.path.isdir(subfolder_path):
            continue  # skip files

        print(f"Processing subfolder: {subfolder}")

        zip_path = os.path.join(subfolder_path, "eagle.zip")
        if os.path.isfile(zip_path):
            try:
                with zipfile.ZipFile(zip_path, "r") as zip_ref:
                    zip_ref.extractall(subfolder_path)
            except zipfile.BadZipFile:
                print(f"  Error: Corrupt ZIP -> {zip_path}")
            except Exception as e:
                print(f"  Error extracting {zip_path}: {e}")

            # After unzip, check for .sch in the subfolder
            sch_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith(".sch")]

            if not sch_files:
                # Look one level deeper for newly created folders
                new_folders = [
                    f for f in os.listdir(subfolder_path)
                    if os.path.isdir(os.path.join(subfolder_path, f))
                ]
                # Ignore macOS resource folder
                new_folders = [f for f in new_folders if f != "__MACOSX"]

                if new_folders:
                    print(f"  No .sch at top-level, found nested folder(s): {new_folders}")
                    # Move .sch and .brd up one level
                    moved_any = False
                    for nf in new_folders:
                        nf_path = os.path.join(subfolder_path, nf)
                        for file in os.listdir(nf_path):
                            if file.lower().endswith((".sch", ".brd")):
                                src = os.path.join(nf_path, file)
                                dst = os.path.join(subfolder_path, file)
                                # Avoid overwrite: add ##N if needed
                                if os.path.exists(dst):
                                    name, ext = os.path.splitext(file)
                                    counter = 1
                                    while os.path.exists(dst):
                                        dst = os.path.join(subfolder_path, f"{name}##{counter}{ext}")
                                        counter += 1
                                try:
                                    os.rename(src, dst)
                                    moved_any = True
                                    print(f"  Moved {file} from {nf_path} -> {dst}")
                                except Exception as e:
                                    print(f"  Failed to move {file}: {e}")
                    if not moved_any:
                        print(f"  Note: No .sch/.brd found in nested folders for {subfolder_path}")
                else:
                    print(f"  No .sch file found in: {subfolder_path}")
        else:
            print(f"  No eagle.zip in: {subfolder_path}")

        # Post-processing: count .sch / .brd now present at subfolder root
        try:
            entries = os.listdir(subfolder_path)
        except Exception as e:
            print(f"  Skipping (cannot list): {subfolder_path} ({e})")
            print("-" * 30)
            continue

        sch_files = [f for f in entries if f.lower().endswith(".sch")]
        brd_files = [f for f in entries if f.lower().endswith(".brd")]

        if len(sch_files) == 1 and len(brd_files) == 1:
            if clean_when_single_pair:
                keep = {sch_files[0], brd_files[0]}
                for item in entries:
                    if item in keep:
                        continue
                    item_path = os.path.join(subfolder_path, item)
                    try:
                        if os.path.isfile(item_path) or os.path.islink(item_path):
                            os.remove(item_path)
                        elif os.path.isdir(item_path):
                            shutil.rmtree(item_path)
                        # print(f"  Deleted: {item_path}")  # uncomment if you want verbose cleanup logs
                    except Exception as e:
                        print(f"  Failed to delete {item_path}: {e}")
        else:
            print(f"  Warning: Expected exactly one .sch and one .brd in {subfolder_path}, "
                  f"found {len(sch_files)} .sch and {len(brd_files)} .brd")

        # Rename .sch/.brd to match the subfolder name; avoid overwriting by suffixing
        for ext in (".sch", ".brd"):
            files = [f for f in os.listdir(subfolder_path) if f.lower().endswith(ext)]
            for f in files:
                wanted = f"{subfolder}{ext}"
                src = os.path.join(subfolder_path, f)
                dst = os.path.join(subfolder_path, wanted)
                if f == wanted:
                    continue
                if os.path.exists(dst):
                    # add ##N to avoid clobbering
                    base = f"{subfolder}"
                    counter = 1
                    while os.path.exists(dst):
                        dst = os.path.join(subfolder_path, f"{base}##{counter}{ext}")
                        counter += 1
                try:
                    os.rename(src, dst)
                    # print(f"  Renamed {f} -> {os.path.basename(dst)}")  # optional log
                except Exception as e:
                    print(f"  Failed to rename {f}: {e}")

        print("-" * 30)

def check_sch_brd_in_subfolders(folder_path):
    """
    Checks each subfolder in folder_path for exactly one .sch and one .brd file,
    both named the same as the subfolder. Prints 'pass' if true, else 'no' and saves fail cases.
    Returns a list of failed subfolders.
    """
    fail_cases = []
    for subfolder in os.listdir(folder_path):
        subfolder_path = os.path.join(folder_path, subfolder)
        if os.path.isdir(subfolder_path):
            sch_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith('.sch')]
            brd_files = [f for f in os.listdir(subfolder_path) if f.lower().endswith('.brd')]
            if len(sch_files) == 1 and len(brd_files) == 1 and sch_files[0] == f"{subfolder}.sch" and brd_files[0] == f"{subfolder}.brd":
                pass
            else:
                fail_cases.append(subfolder)
    return fail_cases


def copy_all_sch_files(source_folder, destination_folder):
    """
    Copies all .sch files inside subfolders of source_folder to destination_folder.
    If the file already exists in destination_folder, it will not be copied.
    Returns a tuple: (count_copied, count_skipped)
    """
    os.makedirs(destination_folder, exist_ok=True)
    count_copied = 0
    count_skipped = 0
    for subfolder in os.listdir(source_folder):
        subfolder_path = os.path.join(source_folder, subfolder)
        if os.path.isdir(subfolder_path):
            for file in os.listdir(subfolder_path):
                if file.lower().endswith('.sch'):
                    src_file = os.path.join(subfolder_path, file)
                    dst_file = os.path.join(destination_folder, file)
                    if not os.path.exists(dst_file):
                        shutil.copy2(src_file, dst_file)
                        # print(f"Copied: {src_file} -> {dst_file}")
                        count_copied += 1
                    else:
                        # print(f"Skipped (already exists): {dst_file}")
                        count_skipped += 1
    print(f"Total copied: {count_copied}, Total skipped: {count_skipped}")
    return count_copied, count_skipped

# Example

In [ ]:
# Example usage:
source_folder_path = r"/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Tutorial Zip"
print(check_zip_in_subfolders(source_folder_path))

# unzip_eagle_zips_in_subfolders(source_folder_path)


# fail_list = check_sch_brd_in_subfolders(folder_path)
# print("No Fail cases:", fail_list)


# Move all .sch files from subfolders to a single destination folder
source_folder_path = r"/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Tutorial Zip"
destination_folder_path = r"/Users/taitinglu/Desktop/IMG2SCH/Sparkfun Tutorial UnZip"
copy_all_sch_files(source_folder_path, destination_folder_path)

# Process full sch

In [19]:
def check_eagle_version(file_path, target_version="9.6.2"):
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            if 'eagle version' in line:
                if f'version="{target_version}"' in line:
                    print(f"File is already in Eagle version {target_version}")
                    return True
                else:
                    print(f"File is in a different version: {line.strip()}")
                    print("Please open and save it in Eagle version 9.6.2.")
                    return False
    print("Version info not found.")
    return False


def convert_sch_folder_to_images(sch_folder, img_folder,fail_csv):
    """
    Converts all .sch files in sch_folder to images in img_folder using visualize_schematic.
    Skips files that fail and saves their names and errors to a list.
    """
    os.makedirs(img_folder, exist_ok=True)
    sch_files = sorted([f for f in os.listdir(sch_folder) if f.lower().endswith('.sch')])[:-1]
    failed_files = []
    for idx, sch_file in enumerate(sch_files):
        sch_path = os.path.join(sch_folder, sch_file)
        img_path = os.path.join(img_folder, f"{os.path.splitext(sch_file)[0]}.png")
        print(f"[{idx}] Processing: {sch_file}")
        try:
            if check_eagle_version(sch_path):
                new_path = img_path.replace(".png", "#0.png")
                if not os.path.exists(new_path):
                    visualize_ful_schematic(sch_path, save_path=img_path, draw_grid=False,draw_bounding_bbox=True)
                else:
                    print(f"Image already exists, skipping.")
                    continue  # Skip if image already exists
            else:
                print(f"Skipping {sch_file} due to version mismatch.")
                failed_files.append((sch_file, "version mismatch"))
        except Exception as e:
            print(f"Failed to process {sch_file}: {e}")
            failed_files.append((sch_file, str(e)))
        
        print("-----------------------")
        
    csv_path = os.path.join(img_folder, fail_csv)
    with open(csv_path, "w", newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(["File", "Error"])
        writer.writerows(failed_files)
    print(f"Failed files saved to: {csv_path}")
    return failed_files

# Example

In [20]:
# Example usage:
sch_folder = r"/Users/taitinglu/Desktop/IMG2SCH/Sch Cleaned"
img_folder = r"/Users/taitinglu/Desktop/IMG2SCH/Img Cleaned"
fail_csv = r"/Users/taitinglu/Desktop/IMG2SCH/failed_sch.csv"
sch_to_img_check = convert_sch_folder_to_images(sch_folder, img_folder, fail_csv)

[0] Processing: 0.5mm 4-pin.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[1] Processing: 1.0mm + 0.5mm FPC stick.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[2] Processing: 12-pin SOIC+TSSOP.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[3] Processing: 14-pin SOIC+TSSOP.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[4] Processing: 16-pin SOIC+TSSOP.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[5] Processing: 16304-TMP102-Qwiic_EagleFiles##1.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[6] Processing: 16400_SparkFun_MicroMod_Machine_Learning_Carrier_Board_EagleFiles.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[7] Processing: 16x8 mono LED FeatherWing rev A.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[8] Processing: 17712_STM32_Thing_Plus_Eagle_Files.

/var/folders/6_/fhplfkl93ts3d6yt6kdrkpc00000gn/T/ipykernel_19978/2588658396.py:66: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, ax = plt.subplots(figsize=(fig_w, fig_h), dpi=100)  # Create figure and axes


Failed to process Adafruit FONA 800 Shield.sch: string indices must be integers, not 'str'
-----------------------
[269] Processing: Adafruit FONA SIM5320 3G Breakout.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[270] Processing: Adafruit FPC Breakout - Pi 5 DSI - RP2350 HSTX.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[271] Processing: Adafruit FT232H Original.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[272] Processing: Adafruit FT232H.sch
File is already in Eagle version 9.6.2
Image already exists, skipping.
[273] Processing: Adafruit FTDI Friend Rev A.sch
File is already in Eagle version 9.6.2
Failed to process Adafruit FTDI Friend Rev A.sch: not enough values to unpack (expected 3, got 2)
-----------------------
[274] Processing: Adafruit FTDI Friend Rev B.sch
File is already in Eagle version 9.6.2
Failed to process Adafruit FTDI Friend Rev B.sch: not enough values to unpack (expected 3, go